<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/1_2_Multiple_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part I. Continuous Response. Multiple Linear Regression

# 2. Множественная линейная регрессия (часть I)

В предыдущей главе модель $y = w_0 + w_1 x + \varepsilon$ описывала прямую на плоскости. Мы научились проводить оптимальную прямую через наблюдаемые точки, осознав, что один-единственный признак — например, площадь квартиры — не исчерпывает всей вариации целевой переменной. В действительности факторов, влияющих на отклик, всегда несколько: площадь важна, но цена также зависит от этажа, удалённости от метро, возраста дома и многих других характеристик. Теперь признаков становится много, и прямая превращается в гиперплоскость в многомерном пространстве. Все идеи, освоенные на простейшем примере, — метод наименьших квадратов, принцип максимального правдоподобия, матричная запись — естественно обобщаются на этот многомерный случай. Более того, вводя понятие базисных функций, мы получаем возможность описывать нелинейные зависимости, оставаясь в рамках чрезвычайно удобной линейной по параметрам модели.

## 1. Постановка задачи множественной линейной регрессии

Пусть для каждого объекта наблюдения доступны значения $D$ признаков, собранные в вектор $\mathbf{x} = (x_1, x_2, \dots, x_D)^T \in \mathbb{R}^D$. Целевая переменная (отклик) по-прежнему является вещественным числом; для единообразия с классической литературой будем обозначать её через $t$ (от *target*). Предполагается, что связь между признаками и откликом описывается линейной моделью

$$
t = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_D x_D + \varepsilon, \tag{1.1}
$$

где  
$w_0$ — свободный член (bias, intercept), определяющий вертикальный сдвиг гиперплоскости;  
$w_j$ ($j = 1,\dots,D$) — весовые коэффициенты, соответствующие отдельным признакам;  
$\varepsilon$ — случайная ошибка, агрегирующая влияние всех неучтённых факторов.

Как и ранее, мы полагаем, что ошибка имеет нулевое математическое ожидание и нормальное распределение с постоянной дисперсией:

$$
\varepsilon \sim \mathcal{N}(0, \sigma^2). \tag{1.2}
$$

Из этого немедленно следует, что условное распределение отклика при заданном векторе признаков также нормально:

$$
p(t \mid \mathbf{x}, \mathbf{w}) = \mathcal{N}\!\left(w_0 + \sum_{j=1}^{D} w_j x_j,\; \sigma^2\right). \tag{1.3}
$$

Иными словами, для любого фиксированного набора признаков $\mathbf{x}$ наиболее вероятное значение $t$ равно линейной комбинации $w_0 + \sum w_j x_j$, а разброс вокруг этого значения постоянен для всех объектов (гомоскедастичность).

Цель состоит в том, чтобы по обучающей выборке из $n$ независимых наблюдений

$$
\mathcal{D} = \{(\mathbf{x}^{(i)}, t^{(i)})\}_{i=1}^n,
$$

найти вектор параметров $\mathbf{w} = (w_0, w_1, \dots, w_D)^T$, наилучшим образом описывающий данные. «Наилучшим» мы, как и в случае простой регрессии, считаем такой вектор, который минимизирует сумму квадратов ошибок предсказания.

Существенное преимущество линейной модели — её интерпретируемость. Знак и абсолютная величина коэффициента $w_j$ имеют прямой содержательный смысл: при увеличении $j$-го признака на одну единицу (и неизменных значениях остальных признаков) отклик в среднем изменяется на $w_j$ единиц. Это свойство делает модель прозрачной и позволяет не только предсказывать, но и объяснять.

> **Углублённый взгляд.** Интерпретация «при фиксации всех остальных» является строгой только при отсутствии корреляции между признаками. Если же признаки сильно скоррелированы (мультиколлинеарность), оценка отдельного коэффициента $w_j$ становится неустойчивой и утрачивает причинно-следственную интерпретацию. Методам диагностики мультиколлинеарности будет посвящён специальный раздел во второй половине главы.

## 2. Матричная запись модели

Оперировать суммами в (1.1) при большом числе признаков неудобно. Матричная запись, введённая для простой регрессии, естественно обобщается на многомерный случай.

Соберём все отклики в вектор-столбец $\mathbf{t} \in \mathbb{R}^n$, а все весовые коэффициенты — в вектор $\mathbf{w} \in \mathbb{R}^{D+1}$ (дополненный $w_0$):

$$
\mathbf{t} = \begin{pmatrix} t^{(1)} \\ t^{(2)} \\ \vdots \\ t^{(n)} \end{pmatrix}, \qquad
\mathbf{w} = \begin{pmatrix} w_0 \\ w_1 \\ \vdots \\ w_D \end{pmatrix}.
$$

Введём **матрицу плана** (design matrix) $X$ размера $n \times (D+1)$. Её первый столбец состоит из единиц и отвечает за свободный член $w_0$, а каждый последующий столбец содержит значения одного из $D$ признаков для всех $n$ наблюдений:

$$
X = \begin{pmatrix}
1 & x_1^{(1)} & x_2^{(1)} & \dots & x_D^{(1)} \\
1 & x_1^{(2)} & x_2^{(2)} & \dots & x_D^{(2)} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_1^{(n)} & x_2^{(n)} & \dots & x_D^{(n)}
\end{pmatrix}.
$$

Тогда система $n$ уравнений (1.1) записывается компактно:

$$
\mathbf{t} = X\mathbf{w} + \boldsymbol{\varepsilon}, \tag{2.1}
$$

где $\boldsymbol{\varepsilon} = (\varepsilon_1, \dots, \varepsilon_n)^T$ — вектор случайных ошибок. Такая форма сразу подсказывает, что оценивание $\mathbf{w}$ может быть выполнено методами линейной алгебры.





**Практический пример: создание матрицы плана в Python.**


In [ ]:
import numpy as np

# Исходные признаки (5 объектов, 2 признака)
X_raw = np.array([[2.5, 3.0],
                  [3.0, 3.5],
                  [4.0, 4.5],
                  [4.5, 5.0],
                  [5.0, 5.5]])

# Добавляем столбец единиц для учёта свободного члена w0
Phi = np.hstack([np.ones((X_raw.shape[0], 1)), X_raw])

print("Матрица плана Phi:")
print(Phi)
print(f"\nРазмерность: {Phi.shape} (n={Phi.shape[0]}, M={Phi.shape[1]})")



## 3. Базисные функции

Модель (2.1) линейна не только по параметрам $\mathbf{w}$, но и по самим исходным признакам. Однако во многих прикладных задачах зависимость отклика от признаков носит существенно нелинейный характер. Цена квартиры может возрастать с площадью не равномерно, а с насыщением; прочность сплава зависит от концентрации примеси по параболическому закону; спрос на товар обнаруживает сезонные всплески, привязанные к определённым календарным датам.

Удачным компромиссом являются **базисные функции**. Идея состоит в том, чтобы заменить исходный вектор признаков $\mathbf{x}$ вектором нелинейных преобразований

$$
\boldsymbol{\phi}(\mathbf{x}) = \bigl(\phi_0(\mathbf{x}), \phi_1(\mathbf{x}), \dots, \phi_{M-1}(\mathbf{x})\bigr)^T,
$$

а модель по-прежнему строить линейной по параметрам:

$$
y(\mathbf{x}, \mathbf{w}) = \sum_{j=0}^{M-1} w_j \phi_j(\mathbf{x}) = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}). \tag{3.1}
$$

Здесь $M$ — количество базисных функций; $\phi_0(\mathbf{x}) \equiv 1$ обеспечивает наличие свободного члена. **Модель остаётся линейной по $\mathbf{w}$, но способна описывать весьма сложные нелинейные зависимости.** Вся математическая техника, развитая для линейной регрессии, сохраняется — достаточно заменить исходную матрицу $X$ на матрицу $\Phi$, составленную из значений базисных функций:

$$
\Phi = \begin{pmatrix}
\phi_0(\mathbf{x}^{(1)}) & \phi_1(\mathbf{x}^{(1)}) & \dots & \phi_{M-1}(\mathbf{x}^{(1)}) \\
\phi_0(\mathbf{x}^{(2)}) & \phi_1(\mathbf{x}^{(2)}) & \dots & \phi_{M-1}(\mathbf{x}^{(2)}) \\
\vdots & \vdots & \ddots & \vdots \\
\phi_0(\mathbf{x}^{(n)}) & \phi_1(\mathbf{x}^{(n)}) & \dots & \phi_{M-1}(\mathbf{x}^{(n)})
\end{pmatrix}.
$$

Рассмотрим три характерных примера базисных функций.

**Полиномиальные признаки.** Если предполагается, что отклик зависит от одного признака $x$ полиномиально, в качестве базисных функций выбирают его степени: $\phi_k(x) = x^k$, $k = 0,1,\dots,K$. Модель превращается в полиномиальную регрессию степени $K$. При $K=1$ получаем обычную прямую; при больших $K$ модель может гибко следовать за данными, изгибаясь в соответствии с наблюдаемой тенденцией.

**Радиально-базисные функции (RBF).** Эти функции имеют вид колоколов, «надетых» на определённые точки пространства признаков:

$$
\phi_j(\mathbf{x}) = \exp\!\left(-\frac{\|\mathbf{x} - \boldsymbol{\mu}_j\|^2}{2\sigma^2}\right),
$$

где $\boldsymbol{\mu}_j$ — центр $j$-го колокола, а $\sigma$ — параметр ширины. Если объект находится вблизи центра $\boldsymbol{\mu}_j$, значение функции близко к единице; при удалении оно быстро убывает до нуля. Комбинируя множество таких локализованных функций, можно аппроксимировать практически любую гладкую поверхность.

**Конструирование признаков вручную.** На практике специалист часто самостоятельно формирует осмысленные нелинейные комбинации. Предположим, в нашем распоряжении два признака: доход клиента $x_1$ и сумма текущего кредита $x_2$. Исходя из экономической логики, можно ожидать, что целевая переменная (например, вероятность дефолта) зависит не только от самих величин, но и от их отношения, а также от того, превышает ли долг доход. Тогда модель может быть записана как

$$
t \approx w_1 x_1 + w_2 x_2 + w_3 \log x_1 + w_4 \,\text{sgn}(x_1 x_2) + w_0.
$$

Здесь добавлены базисные функции $\phi_3 = \log x_1$ и $\phi_4 = \text{sgn}(x_1 x_2)$. Хотя исходных признаков всего два, после такого преобразования мы решаем четырёхмерную линейную регрессию — подход одновременно гибкий и интерпретируемый.

**Мостик к категориальным признакам.** Если признак является не числовым, а категориальным (например, порода питомца или марка автомобиля), его также можно представить набором индикаторных базисных функций: для каждой возможной категории вводится функция, равная единице при совпадении и нулю в остальных случаях. Возникающая при этом линейная зависимость индикаторов требует отдельного обсуждения; она будет подробно рассмотрена в разделе, посвящённом подготовке признаков. Здесь достаточно понимать, что с помощью базисных функций любые данные — числовые, нелинейно-преобразованные и категориальные — могут быть приведены к единой линейной модели.

Везде далее мы используем обозначение $\Phi$ для матрицы плана, подразумевая, что она может содержать как исходные признаки, так и их базисные преобразования.

## 4. Метод наименьших квадратов и вероятностное обоснование

Имея матрицу $\Phi$, определим функцию потерь — сумму квадратов ошибок. Для удобства вычислений введём в неё множитель $1/2$, не изменяющий положения минимума:

$$
E(\mathbf{w}) = \frac{1}{2} \sum_{i=1}^{n} \bigl(t^{(i)} - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}^{(i)})\bigr)^2. \tag{4.1}
$$

Множитель $1/2$ сократится при дифференцировании и сделает последующие формулы более компактными.

Установим связь между минимизацией $E(\mathbf{w})$ и вероятностной моделью. Пусть ошибки $\varepsilon_i$ независимы и одинаково распределены:

$$
\varepsilon_i \sim \mathcal{N}(0, \sigma^2),\qquad i = 1,\dots,n.
$$

Тогда условная плотность каждого наблюдения есть

$$
p(t^{(i)} \mid \mathbf{x}^{(i)}, \mathbf{w}) = \mathcal{N}\bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}^{(i)}),\, \sigma^2\bigr).
$$

В силу независимости наблюдений функция правдоподобия всей выборки равна произведению индивидуальных плотностей:

$$
L(\mathbf{w}) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{\bigl(t^{(i)} - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}^{(i)})\bigr)^2}{2\sigma^2}\right). \tag{4.2}
$$

Перейдём к логарифмической функции правдоподобия — монотонному преобразованию, не смещающему точку максимума:

$$
\log L(\mathbf{w}) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{n} \bigl(t^{(i)} - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}^{(i)})\bigr)^2. \tag{4.3}
$$

Первое слагаемое не зависит от $\mathbf{w}$, а второе с точностью до константы представляет собой $-E(\mathbf{w})$. Таким образом, максимизация логарифма правдоподобия эквивалентна минимизации суммы квадратов ошибок. Следовательно, оценки метода наименьших квадратов $\hat{\mathbf{w}}$ совпадают с оценками максимального правдоподобия при условии нормальности и гомоскедастичности ошибок. Этот результат полностью аналогичен выводу, полученному для простой регрессии, и подчёркивает, что МНК является не эвристическим приёмом, а прямым следствием фундаментального статистического принципа.

> **Углублённый взгляд: MLE для дисперсии ошибки.**  
> Логарифм правдоподобия (4.3) зависит также от $\sigma^2$. Продифференцировав $\log L$ по $\sigma^2$ и приравняв производную к нулю, получаем
> $$
> \frac{\partial \log L}{\partial \sigma^2} = -\frac{n}{2}\frac{1}{\sigma^2} + \frac{1}{2\sigma^4}\sum_{i=1}^{n} \bigl(t^{(i)} - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}^{(i)})\bigr)^2 = 0.
> $$
> Умножая на $2\sigma^4$, приходим к $-n\sigma^2 + SSE = 0$, откуда
> $$
> \hat{\sigma}^2_{\text{MLE}} = \frac{SSE}{n} = \frac{1}{n}\sum_{i=1}^{n} \bigl(t^{(i)} - \hat{t}^{(i)}\bigr)^2.
> $$
> Эта оценка является смещённой. Несмещённая оценка дисперсии ошибок имеет вид
> $$
> s^2 = \frac{SSE}{n - M},
> $$
> где $M$ — количество параметров модели (включая свободный член). Потеря $M$ степеней свободы объясняется тем, что $M$ чисел уже использованы для оценивания самого вектора $\mathbf{w}$. Именно $s^2$ применяется при построении доверительных интервалов и проверке статистических гипотез относительно коэффициентов.

Итак, располагая функцией потерь и её вероятностным обоснованием, мы готовы перейти к отысканию оптимального вектора весов $\mathbf{w}$. В следующей части будут выведены нормальные уравнения в матричной форме, получено явное решение $\hat{\mathbf{w}} = (\Phi^T \Phi)^{-1} \Phi^T \mathbf{t}$ и рассмотрены его геометрические и статистические свойства.



## 5. Градиент целевой функции

Рассматриваемая функция потерь имеет вид

$$
E(\mathbf{w}) = \frac{1}{2} \sum_{i=1}^{n} \bigl(t_i - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr)^2,
$$

где для сокращения записи принято $t_i \equiv t^{(i)}$, $\mathbf{x}_i \equiv \mathbf{x}^{(i)}$. Функция $E(\mathbf{w})$ является выпуклой квадратичной формой; её минимум может быть найден приравниванием градиента к нулю. Вычислим градиент, т.е. вектор частных производных по всем компонентам $\mathbf{w} = (w_0, w_1, \dots, w_{M-1})^T$.

Для произвольного $k = 0,\dots,M-1$ имеем

$$
\frac{\partial E}{\partial w_k}
= \frac{\partial}{\partial w_k} \left( \frac{1}{2} \sum_{i=1}^{n} \bigl(t_i - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr)^2 \right).
$$

Применяя правило цепочки и учитывая, что производная квадрата с коэффициентом $1/2$ уничтожает возникающую двойку, получаем

$$
\frac{\partial E}{\partial w_k}
= \sum_{i=1}^{n} \bigl(t_i - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr)
\cdot \frac{\partial}{\partial w_k} \bigl(t_i - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr).
$$

Внутренняя производная вычисляется с учётом разложения скалярного произведения:

$$
\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) = \sum_{j=0}^{M-1} w_j \phi_j(\mathbf{x}_i),
\qquad
\frac{\partial}{\partial w_k} \bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr) = \phi_k(\mathbf{x}_i),
$$

откуда

$$
\frac{\partial}{\partial w_k} \bigl(t_i - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr) = -\phi_k(\mathbf{x}_i).
$$

Следовательно,

$$
\frac{\partial E}{\partial w_k}
= \sum_{i=1}^{n} \bigl(t_i - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr) \cdot \bigl(-\phi_k(\mathbf{x}_i)\bigr).
$$

Замечая, что $t_i - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) = -(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i)$, приходим к компактному выражению

$$
\boxed{\frac{\partial E}{\partial w_k} = \sum_{i=1}^{n} \bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i\bigr)\,\phi_k(\mathbf{x}_i)}. \tag{5.1}
$$

Теперь соберём все частные производные в вектор-столбец градиента:

$$
\nabla_{\mathbf{w}} E =
\begin{pmatrix}
\frac{\partial E}{\partial w_0} \\
\frac{\partial E}{\partial w_1} \\
\vdots \\
\frac{\partial E}{\partial w_{M-1}}
\end{pmatrix}
=
\begin{pmatrix}
\sum_{i=1}^{n} (\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i)\,\phi_0(\mathbf{x}_i) \\
\sum_{i=1}^{n} (\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i)\,\phi_1(\mathbf{x}_i) \\
\vdots \\
\sum_{i=1}^{n} (\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i)\,\phi_{M-1}(\mathbf{x}_i)
\end{pmatrix}.
$$

Заметим, что каждая компонента представляет собой сумму по всем наблюдениям произведений ошибки предсказания на значение соответствующей базисной функции. Эту конструкцию можно записать как одну сумму векторов:

$$
\boxed{\nabla_{\mathbf{w}} E = \sum_{i=1}^{n} \bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i\bigr)\,\boldsymbol{\phi}(\mathbf{x}_i)}. \tag{5.2}
$$

Для перехода к матричной форме введём матрицу плана $\Phi \in \mathbb{R}^{n \times M}$, вектор целевых значений $\mathbf{t} \in \mathbb{R}^n$, вектор предсказаний $\hat{\mathbf{t}} = \Phi \mathbf{w}$ и вектор ошибок (остатков) $\mathbf{e} = \Phi \mathbf{w} - \mathbf{t}$. Тогда $i$-я компонента ошибки есть $e_i = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i$. Рассмотрим произведение $\Phi^T \mathbf{e}$. Его $j$-я компонента равна

$$
(\Phi^T \mathbf{e})_j = \sum_{i=1}^{n} \Phi_{ij} e_i = \sum_{i=1}^{n} \phi_j(\mathbf{x}_i) e_i,
$$

что в точности совпадает с (5.1). Таким образом,

$$
\boxed{\nabla_{\mathbf{w}} E = \Phi^T (\Phi \mathbf{w} - \mathbf{t})}. \tag{5.3}
$$

**Иллюстративная проверка.** Пусть $n = 2$, $M = 3$ (включая константную функцию $\phi_0 = 1$). Зададим

$$
\Phi = \begin{pmatrix}
1 & 2 & 3 \\
1 & 4 & 1
\end{pmatrix}, \quad
\mathbf{w} = \begin{pmatrix} 1 \\ 0 \\ 2 \end{pmatrix}, \quad
\mathbf{t} = \begin{pmatrix} 5 \\ 6 \end{pmatrix}.
$$

Тогда $\hat{\mathbf{t}} = \Phi \mathbf{w} = \begin{pmatrix} 7 \\ 3 \end{pmatrix}$, $\mathbf{e} = \begin{pmatrix} 2 \\ -3 \end{pmatrix}$. Произведение $\Phi^T \mathbf{e}$ даёт

$$
\Phi^T \mathbf{e} = \begin{pmatrix}
1 & 1 \\
2 & 4 \\
3 & 1
\end{pmatrix}
\begin{pmatrix} 2 \\ -3 \end{pmatrix} =
\begin{pmatrix} -1 \\ -8 \\ 3 \end{pmatrix}.
$$

Непосредственное вычисление компонент градиента по (5.2) приводит к тем же значениям:
$k=0:\; 2\cdot1 + (-3)\cdot1 = -1,$
$k=1:\; 2\cdot2 + (-3)\cdot4 = -8,$
$k=2:\; 2\cdot3 + (-3)\cdot1 = 3.$

Совпадение подтверждает корректность матричной записи.

## 6. Нормальные уравнения и свойства МНК-оценок

Приравнивая градиент (5.3) к нулю, получаем условие стационарности

$$
\Phi^T (\Phi \mathbf{w} - \mathbf{t}) = \mathbf{0},
$$

которое эквивалентно системе **нормальных уравнений**

$$
\Phi^T \Phi \mathbf{w} = \Phi^T \mathbf{t}. \tag{6.1}
$$

Матрица $\Phi^T \Phi \in \mathbb{R}^{M \times M}$ симметрична и неотрицательно определена. Если она положительно определена, что эквивалентно линейной независимости столбцов $\Phi$ (отсутствию совершенной мультиколлинеарности), то существует единственное решение

$$
\boxed{\hat{\mathbf{w}} = (\Phi^T \Phi)^{-1} \Phi^T \mathbf{t}}. \tag{6.2}
$$

Это прямое обобщение результата для простой линейной регрессии. Вектор предсказаний $\hat{\mathbf{t}} = \Phi \hat{\mathbf{w}}$ при этом является ортогональной проекцией $\mathbf{t}$ на подпространство, порождённое столбцами матрицы $\Phi$. Действительно, матрица

$$
P = \Phi (\Phi^T \Phi)^{-1} \Phi^T \in \mathbb{R}^{n \times n}
$$

является оператором ортогонального проектирования (проектором): она симметрична ($P^T = P$), идемпотентна ($P^2 = P$) и обладает свойствами $P\Phi = \Phi$, $(I-P)\Phi = 0$. Тогда $\hat{\mathbf{t}} = P\mathbf{t}$, а вектор остатков $\mathbf{e} = \mathbf{t} - \hat{\mathbf{t}} = (I-P)\mathbf{t}$ ортогонален столбцам $\Phi$, что немедленно следует из $\Phi^T\mathbf{e} = \Phi^T(I-P)\mathbf{t} = \mathbf{0}$.

Статистические свойства оценки $\hat{\mathbf{w}}$. При предположениях классической линейной модели ($\mathbb{E}[\boldsymbol{\varepsilon}] = \mathbf{0}$, $\mathrm{Var}(\boldsymbol{\varepsilon}) = \sigma^2 I$, независимость ошибок) оценка $\hat{\mathbf{w}}$ является несмещённой:

$$
\mathbb{E}[\hat{\mathbf{w}}] = (\Phi^T \Phi)^{-1} \Phi^T \mathbb{E}[\mathbf{t}] = (\Phi^T \Phi)^{-1} \Phi^T \Phi \mathbf{w} = \mathbf{w}.
$$

Её ковариационная матрица имеет вид

$$
\mathrm{Var}(\hat{\mathbf{w}}) = (\Phi^T \Phi)^{-1} \Phi^T \mathrm{Var}(\mathbf{t}) \Phi (\Phi^T \Phi)^{-1} = \sigma^2 (\Phi^T \Phi)^{-1}. \tag{6.3}
$$

Диагональные элементы этой матрицы дают дисперсии отдельных коэффициентов $\mathrm{Var}(\hat{w}_j) = \sigma^2 [(\Phi^T \Phi)^{-1}]_{jj}$. В сочетании с нормальностью ошибок отсюда следуют все стандартные тесты (t-тест для отдельного коэффициента, F-тест для группы), которые будут подробно рассмотрены во второй половине главы.


Ты прав, в предыдущем итоговом варианте изложение было несколько сжато. Я восстановлю и дополню твои вкладки максимально подробно, в строгом научном стиле, чтобы этот углублённый взгляд стал полноценным мини-разделом, который можно читать как самостоятельное доказательство.

Ниже — переработанный и расширенный фрагмент, который включает все твои шаги: от транспонирования градиента до явного покомпонентного отождествления сумм с матрицами $\Phi^T\Phi$ и $\mathbf{t}^T\Phi$, с комментариями о геометрической сути.

---

> **Углублённый взгляд: альтернативный вывод нормальных уравнений.**  
> Тот же результат можно получить, стартуя с транспонированной формы градиента. Из (5.2) имеем
> $$
> \nabla_{\mathbf{w}} E = \sum_{i=1}^{n} e_i \boldsymbol{\phi}(\mathbf{x}_i),
> $$
> где ошибка $e_i = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i$. Транспонируем обе стороны:
> $$
> (\nabla_{\mathbf{w}} E)^T = \sum_{i=1}^{n} e_i \boldsymbol{\phi}(\mathbf{x}_i)^T.
> $$
> Подставляя явное выражение для $e_i$, получаем
> $$
> (\nabla_{\mathbf{w}} E)^T = \sum_{i=1}^{n} \bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i\bigr) \boldsymbol{\phi}(\mathbf{x}_i)^T.
> $$
> Разобьём сумму на две:
> $$
> (\nabla_{\mathbf{w}} E)^T = \underbrace{\sum_{i=1}^{n} \bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr) \boldsymbol{\phi}(\mathbf{x}_i)^T}_{S_1} - \underbrace{\sum_{i=1}^{n} t_i \boldsymbol{\phi}(\mathbf{x}_i)^T}_{S_2}.
> $$
> Рассмотрим каждое слагаемое подробно.
>
> **Слагаемое $S_1$.**  
> Заметим, что $\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)$ — это скаляр (вещественное число). Поэтому его можно внести внутрь внешнего произведения:
> $$
> \bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i)\bigr) \boldsymbol{\phi}(\mathbf{x}_i)^T
> = \mathbf{w}^T \bigl(\boldsymbol{\phi}(\mathbf{x}_i) \boldsymbol{\phi}(\mathbf{x}_i)^T\bigr).
> $$
> Здесь $\boldsymbol{\phi}(\mathbf{x}_i) \boldsymbol{\phi}(\mathbf{x}_i)^T$ — матрица размера $M \times M$, называемая *внешним произведением* вектора-столбца на самого себя. Её $(j,k)$-й элемент есть произведение $j$-й и $k$-й компонент вектора $\boldsymbol{\phi}(\mathbf{x}_i)$:
> $$
> \bigl(\boldsymbol{\phi}(\mathbf{x}_i) \boldsymbol{\phi}(\mathbf{x}_i)^T\bigr)_{jk}
> = \phi_j(\mathbf{x}_i) \, \phi_k(\mathbf{x}_i).
> $$
> Суммируя по всем $i$, получаем матрицу
> $$
> \sum_{i=1}^{n} \boldsymbol{\phi}(\mathbf{x}_i) \boldsymbol{\phi}(\mathbf{x}_i)^T,
> $$
> у которой $(j,k)$-й элемент равен $\sum_{i=1}^{n} \phi_j(\mathbf{x}_i) \phi_k(\mathbf{x}_i)$. Но в точности такой же вид имеет $(j,k)$-й элемент матрицы $\Phi^T \Phi$:
> $$
> (\Phi^T \Phi)_{jk} = \sum_{i=1}^{n} \Phi_{ij} \Phi_{ik}
> = \sum_{i=1}^{n} \phi_j(\mathbf{x}_i) \phi_k(\mathbf{x}_i),
> $$
> поскольку $\Phi_{ij} = \phi_j(\mathbf{x}_i)$. Следовательно,
> $$
> \sum_{i=1}^{n} \boldsymbol{\phi}(\mathbf{x}_i) \boldsymbol{\phi}(\mathbf{x}_i)^T = \Phi^T \Phi.
> $$
> Тогда
> $$
> S_1 = \mathbf{w}^T \sum_{i=1}^{n} \bigl(\boldsymbol{\phi}(\mathbf{x}_i) \boldsymbol{\phi}(\mathbf{x}_i)^T\bigr)
> = \mathbf{w}^T \Phi^T \Phi.
> $$
> **Слагаемое $S_2$.**  
> Здесь $t_i$ — скаляр, а $\boldsymbol{\phi}(\mathbf{x}_i)^T$ — вектор-строка длины $M$. Их произведение даёт строку $t_i \boldsymbol{\phi}(\mathbf{x}_i)^T$, суммируя которые по всем наблюдениям, получаем вектор-строку
> $$
> S_2 = \sum_{i=1}^{n} t_i \boldsymbol{\phi}(\mathbf{x}_i)^T.
> $$
> $j$-я компонента этой строки есть $\sum_{i=1}^{n} t_i \phi_j(\mathbf{x}_i)$. С другой стороны, $j$-я компонента строки $\mathbf{t}^T \Phi$ равна
> $$
> (\mathbf{t}^T \Phi)_j = \sum_{i=1}^{n} t_i \Phi_{ij} = \sum_{i=1}^{n} t_i \phi_j(\mathbf{x}_i).
> $$
> Следовательно,
> $$
> S_2 = \mathbf{t}^T \Phi.
> $$
> Подставляя $S_1$ и $S_2$ в выражение для $(\nabla_{\mathbf{w}} E)^T$, получаем
> $$
> (\nabla_{\mathbf{w}} E)^T = \mathbf{w}^T \Phi^T \Phi - \mathbf{t}^T \Phi.
> $$
> Приравниваем нулю (условие стационарности):
> $$
> \mathbf{w}^T \Phi^T \Phi - \mathbf{t}^T \Phi = \mathbf{0}^T,
> $$
> откуда
> $$
> \mathbf{w}^T \Phi^T \Phi = \mathbf{t}^T \Phi.
> $$
> Транспонируем обе стороны, используя симметричность $\Phi^T \Phi$ (поскольку $(\Phi^T \Phi)^T = \Phi^T \Phi$), и свойство $(\mathbf{t}^T \Phi)^T = \Phi^T \mathbf{t}$:
> $$
> \Phi^T \Phi \mathbf{w} = \Phi^T \mathbf{t}.
> $$
> Это и есть нормальные уравнения. Решение, если матрица $\Phi^T \Phi$ обратима, даёт
> $$
> \hat{\mathbf{w}} = (\Phi^T \Phi)^{-1} \Phi^T \mathbf{t}.
> $$
> Данный вывод ясно показывает, что условие $\Phi^T (\Phi\mathbf{w} - \mathbf{t}) = \mathbf{0}$ эквивалентно ортогональности вектора остатков $\mathbf{e} = \Phi\mathbf{w} - \mathbf{t}$ всем столбцам матрицы $\Phi$. Тем самым предсказанные значения $\hat{\mathbf{t}} = \Phi\hat{\mathbf{w}}$ являются ортогональной проекцией $\mathbf{t}$ на линейное подпространство, порождённое столбцами $\Phi$. Этот геометрический факт лежит в основе многих дальнейших свойств МНК-оценок (несмещённость, теорема Гаусса–Маркова, распределение остатков).

**Невырожденность $\Phi^T \Phi$.** Матрица $\Phi^T \Phi$ вырождена в точности тогда, когда столбцы $\Phi$ линейно зависимы, т.е. имеет место совершенная мультиколлинеарность. Примером может служить одновременное включение в модель признаков «площадь в квадратных метрах» и «площадь в гектарах»: второй столбец пропорционален первому, и обратная матрица не существует. На практике такая проблема устраняется удалением избыточных признаков либо применением регуляризации (гребневая регрессия, лассо). В вырожденном случае можно также использовать псевдообратную матрицу Мура–Пенроуза, которая даёт решение с минимальной нормой.




## 7. Вычислительная сложность и проблемы «точного» решения

Аналитическое решение $\hat{\mathbf{w}} = (\Phi^T\Phi)^{-1}\Phi^T\mathbf{t}$ изящно, но при больших размерах данных оно сталкивается с двумя главными препятствиями: высокой вычислительной стоимостью и возможной численной неустойчивостью.

**Вычислительная сложность.**  
Формирование матрицы $\Phi^T\Phi$ требует умножения матрицы размера $n \times M$ на $M \times n$, что даёт $O(M^2 n)$ операций. Обращение получившейся $M \times M$ матрицы стандартными методами (Гаусса или Холецкого) добавляет $O(M^3)$. Суммарная сложность — $O(M^2 n + M^3)$. Например, при $M = 1000$ и $n = 10^6$ формирование $\Phi^T\Phi$ занимает около $10^{12}$ операций, а обращение — ещё порядка $10^9$; при $M = 10\,000$ кубическое слагаемое становится неподъёмным.

На практике редко перемножают матрицы в выражении $(\Phi^T\Phi)^{-1}\Phi^T$ целиком; вместо этого сначала вычисляют вектор $\Phi^T\mathbf{t}$, а затем решают систему $(\Phi^T\Phi)\mathbf{w} = \Phi^T\mathbf{t}$. Это экономит память и время. Существуют также более быстрые алгоритмы перемножения матриц (Штрассен, Виноград) и итерационные методы (сопряжённых градиентов), которые позволяют избежать явного обращения.

**Число обусловленности и его геометрический смысл.**  
Даже если матрица $\Phi^T\Phi$ формально обратима, она может быть *плохо обусловленной*. Число обусловленности $\kappa(\Phi)$ определяется через её сингулярные числа как $\kappa(\Phi) = \sigma_{\max}/\sigma_{\min}$. Грубо говоря, оно показывает, насколько различаются масштабы, в которых матрица растягивает векторы по разным направлениям. Если $\kappa$ велико, матрица почти вырождена: малые изменения в правой части $\mathbf{t}$ приводят к катастрофически большим изменениям решения $\hat{\mathbf{w}}$. Более того, погрешность решения системы $(\Phi^T\Phi)\mathbf{w} = \Phi^T\mathbf{t}$ может возрастать пропорционально $\kappa^2(\Phi)$. Поэтому даже умеренное $\kappa \approx 100$ способно «съесть» несколько значащих цифр точности.

Геометрически линии уровня функции потерь $E(\mathbf{w}) = \frac{1}{2}\|\mathbf{t} - \Phi\mathbf{w}\|^2$ представляют собой эллипсоиды. Их форма определяется собственными значениями матрицы $\Phi^T\Phi$. Если собственные значения различаются на порядки, эллипсоиды сильно вытянуты, образуя узкий «овраг». Вдоль длинной оси функция почти постоянна, и положение минимума становится крайне чувствительным к вычислительным погрешностям.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# --- Функция для рисования эллипсоида (линии уровня квадратичной формы) ---
def plot_ellipse(ax, A, title, color='blue'):
    """
    Рисует эллипс {w: w^T A w = 1} для симметричной положительно определённой матрицы A.
    Эллипс показывает линии уровня функции потерь E(w) = 0.5 * w^T A w.
    """
    # Собственные значения и векторы
    eigvals, eigvecs = np.linalg.eigh(A)

    # Полуоси эллипса: 1 / sqrt(собственное значение)
    radii = 1.0 / np.sqrt(eigvals)

    # Угол поворота эллипса (в градусах)
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))

    # Рисуем эллипс
    ellipse = Ellipse(xy=(0, 0), width=2 * radii[0], height=2 * radii[1],
                      angle=angle, facecolor='none', edgecolor=color, linewidth=2)
    ax.add_patch(ellipse)

    # Рисуем оси эллипса (направления собственных векторов)
    for i in range(2):
        vec = eigvecs[:, i] * radii[i]
        ax.arrow(0, 0, vec[0], vec[1], head_width=0.05, head_length=0.05,
                 fc=color, ec=color, alpha=0.5, length_includes_head=True)

    ax.set_aspect('equal')
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.grid(alpha=0.3)
    ax.set_title(title)

# --- Пример 1: хорошо обусловленная матрица (κ ≈ 1) ---
# Эллипсоид близок к окружности — градиентный спуск сходится быстро
A_good = np.array([[1.0, 0.1],
                   [0.1, 1.0]])

# --- Пример 2: плохо обусловленная матрица (κ ≈ 100) ---
# Эллипсоид сильно вытянут — образуется узкий «овраг»,
# градиентный спуск «метается» поперёк оврага и сходится медленно
A_bad = np.array([[1.0, 0.99],
                  [0.99, 1.0]])

# --- Построение графиков ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

plot_ellipse(ax1, A_good,
             f'Хорошо обусловленная матрица\nκ = {np.linalg.cond(A_good):.1f}',
             color='green')

plot_ellipse(ax2, A_bad,
             f'Плохо обусловленная матрица\nκ = {np.linalg.cond(A_bad):.1f}',
             color='red')

plt.suptitle('Линии уровня функции потерь (эллипсоиды) при разном числе обусловленности', fontsize=12)
plt.tight_layout()
plt.show()

# --- Дополнительно: печать числовых значений ---
print(f"Число обусловленности κ (хорошая): {np.linalg.cond(A_good):.2f}")
print(f"Число обусловленности κ (плохая):   {np.linalg.cond(A_bad):.2f}")
print("\nПри κ ≫ 1 эллипсоиды сильно вытянуты — образуется «овраг».")
print("В овраге градиентный спуск сходится медленно, а МНК-решение крайне чувствительно к малым изменениям данных.")


Рассмотрим маленький пример, чтобы ощутить эффект. Пусть

$$
\Phi = \begin{pmatrix}
1 & 1 \\
1 & 1.0001
\end{pmatrix}, \quad \mathbf{t} = \begin{pmatrix} 2 \\ 2.0001 \end{pmatrix}.
$$

Точное решение: $\hat{\mathbf{w}} = (1, 1)^T$. Теперь чуть изменим $\mathbf{t}$ на $\mathbf{t}' = \begin{pmatrix}2 \\ 2\end{pmatrix}$. Тогда решение становится $\hat{\mathbf{w}}' = (2, 0)^T$. Незначительное изменение отклика полностью перевернуло коэффициенты. Причина — почти линейная зависимость столбцов $\Phi$ и, как следствие, большое число обусловленности $\kappa(\Phi) \approx 2\times 10^4$.

> **Углублённый взгляд: методы повышения численной устойчивости.**  
>
> **QR-разложение.**  
> Представим матрицу $\Phi$ в виде $\Phi = QR$, где $Q$ — ортогональная ($Q^TQ = I$), а $R$ — верхнетреугольная. Тогда
> $$
> \hat{\mathbf{w}} = (R^T Q^T Q R)^{-1} R^T Q^T \mathbf{t} = (R^T R)^{-1} R^T Q^T \mathbf{t} = R^{-1} Q^T \mathbf{t}.
> $$
> Вместо обращения $R^T R$ достаточно решить треугольную систему $R\mathbf{w} = Q^T \mathbf{t}$ обратной подстановкой, что требует лишь $O(M^2)$ операций. Число обусловленности $R$ совпадает с $\kappa(\Phi)$, а не с $\kappa^2(\Phi)$, поэтому метод гораздо устойчивее.
>
> *Пример.*  
> Для $\Phi = \begin{pmatrix}1 & 1 \\ 1 & 1.0001\end{pmatrix}$ QR-разложение (полученное, скажем, методом вращений) даёт
> $$
> Q \approx \begin{pmatrix} -0.7071 & -0.7071 \\ -0.7071 & 0.7071 \end{pmatrix}, \quad
> R \approx \begin{pmatrix} -1.4142 & -1.4142 \\ 0 & 0.0000707 \end{pmatrix}.
> $$
> Тогда $Q^T\mathbf{t} \approx \begin{pmatrix}-2.8284 \\ 0.0000707\end{pmatrix}$, и обратная подстановка даёт $w_2 = 1.0$, $w_1 = 1.0$, как и должно быть. Заметьте, что элементы $R$ имеют разный масштаб, но обусловленность $R$ равна $\kappa(\Phi) \approx 2\times10^4$, однако мы избежали образования $\Phi^T\Phi$ с его $\kappa^2$.
>
> **Сингулярное разложение (SVD).**  
> Ещё более универсальный подход: $\Phi = U\Sigma V^T$, где $U,V$ ортогональны, а $\Sigma$ — диагональная матрица с сингулярными числами $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r > 0$. Тогда решение можно записать как
> $$
> \hat{\mathbf{w}} = V \Sigma^{-1} U^T \mathbf{t},
> $$
> где $\Sigma^{-1}$ — диагональная матрица с элементами $1/\sigma_i$, а для $\sigma_i = 0$ берут 0 (псевдообратная). Это даёт решение с минимальной нормой даже при вырожденной $\Phi^T\Phi$. Кроме того, SVD сразу подсказывает идею регуляризации: замена $1/\sigma_i$ на $\sigma_i/(\sigma_i^2 + \lambda)$ приводит к гребневой регрессии (см. раздел «Регуляризация»).
>
> *Пример (продолжение).*  
> Для той же $\Phi$ сингулярные числа: $\sigma_1 \approx 2.00005$, $\sigma_2 \approx 0.00005$. Видно, что $\sigma_2$ очень мало, откуда $\kappa(\Phi) \approx 40000$. Если мы просто обнулим $\sigma_2$ в псевдообратной, получим решение $\hat{\mathbf{w}} = V \Sigma_{\text{pinv}} U^T \mathbf{t}$, которое уже не будет столь чувствительным к шуму, хотя и смещённым. Это и есть суть регуляризации.

**Связь с собственными значениями.**  
Если признаки стандартизованы, то $\Phi^T\Phi$ (с точностью до $1/n$) совпадает с корреляционной матрицей $R$. Тогда число обусловленности $\kappa(R) = \lambda_{\max}/\lambda_{\min}$ измеряет мультиколлинеарность. Большое $\kappa$ означает, что есть направления с очень малой дисперсией, и соответствующие коэффициенты оцениваются ненадёжно.

Таким образом, прямое обращение $\Phi^T\Phi$ чревато численными проблемами, но даже при их решении с помощью QR или SVD остаётся высокая вычислительная сложность для больших $M$. Поэтому в следующей части мы перейдём к итеративным методам, которые не требуют обращения матриц вовсе.




## 8. Численный пример с двумя признаками

Проиллюстрируем вычисления на конкретном примере, не прибегая к программному коду.

Пусть требуется оценить стоимость подержанного автомобиля по его возрасту $x_1$ (лет) и пробегу $x_2$ (тыс. км). Даны три наблюдения:

| Автомобиль | Возраст $x_1$ | Пробег $x_2$ | Цена $t$ (тыс. долл.) |
|------------|----------------|---------------|--------------------------|
| 1          | 3              | 60            | 18                       |
| 2          | 5              | 80            | 14                       |
| 3          | 2              | 40            | 22                       |

Модель: $t = w_0 + w_1 x_1 + w_2 x_2 + \varepsilon$. Матрица плана с единичным столбцом имеет вид

$$
\Phi = \begin{pmatrix}
1 & 3 & 60 \\
1 & 5 & 80 \\
1 & 2 & 40
\end{pmatrix}.
$$

**Шаг 1. Вычисление $\Phi^T \Phi$.** Суммируя необходимые величины,
$$
\sum x_{i1} = 10,\; \sum x_{i2} = 180,\;
\sum x_{i1}^2 = 38,\; \sum x_{i2}^2 = 11600,\;
\sum x_{i1}x_{i2} = 660,
$$
получаем
$$
\Phi^T \Phi = \begin{pmatrix}
3 & 10 & 180 \\
10 & 38 & 660 \\
180 & 660 & 11600
\end{pmatrix}.
$$

**Шаг 2. Вычисление $\Phi^T \mathbf{t}$.**
$$
\mathbf{t} = \begin{pmatrix} 18 \\ 14 \\ 22 \end{pmatrix},\quad
\Phi^T \mathbf{t} = \begin{pmatrix}
18+14+22 \\
3\cdot18 + 5\cdot14 + 2\cdot22 \\
60\cdot18 + 80\cdot14 + 40\cdot22
\end{pmatrix} = \begin{pmatrix} 54 \\ 168 \\ 3080 \end{pmatrix}.
$$

**Шаг 3. Обращение матрицы $\Phi^T \Phi$.** Определитель
$$
\det A = 3(38\cdot11600 - 660^2) - 10(10\cdot11600 - 660\cdot180) + 180(10\cdot660 - 38\cdot180).
$$
Промежуточные значения:
$38\cdot11600 = 440800,\; 660^2 = 435600,\; 5200;$
$10\cdot11600 = 116000,\; 660\cdot180 = 118800,\; -2800;$
$10\cdot660 = 6600,\; 38\cdot180 = 6840,\; -240.$
Отсюда
$$
\det A = 3\cdot5200 - 10\cdot(-2800) + 180\cdot(-240) = 15600 + 28000 - 43200 = 400.
$$
Матрица алгебраических дополнений (симметричная) даёт обратную:
$$
(\Phi^T \Phi)^{-1} = \frac{1}{400}
\begin{pmatrix}
5200 & 2800 & -240 \\
2800 & 2400 & -180 \\
-240 & -180 & 14
\end{pmatrix}.
$$

**Шаг 4. Нахождение $\hat{\mathbf{w}}$.**
$$
\hat{\mathbf{w}} = \frac{1}{400}
\begin{pmatrix}
5200 & 2800 & -240 \\
2800 & 2400 & -180 \\
-240 & -180 & 14
\end{pmatrix}
\begin{pmatrix} 54 \\ 168 \\ 3080 \end{pmatrix}
= \begin{pmatrix}
(5200\cdot54 + 2800\cdot168 - 240\cdot3080)/400 \\
(2800\cdot54 + 2400\cdot168 - 180\cdot3080)/400 \\
(-240\cdot54 - 180\cdot168 + 14\cdot3080)/400
\end{pmatrix}.
$$

Вычисления дают
$$
w_0 = \frac{280800 + 470400 - 739200}{400} = 30,\quad
w_1 = \frac{151200 + 403200 - 554400}{400} = 0,\quad
w_2 = \frac{-12960 - 30240 + 43120}{400} = -0.2.
$$

Итак, $\hat{\mathbf{w}} = (30,\, 0,\, -0.2)^T$, а модель принимает вид
$$
\hat{t} = 30 - 0.2\,x_2.
$$

**Интерпретация.** Нулевое значение коэффициента при возрасте ($w_1=0$) указывает на то, что в рамках данной выборки возраст не продемонстрировал дополнительного влияния на цену при фиксированном пробеге — вероятно, эффект старения уже учтён в пробеге. Свободный член $w_0 = 30$ тыс. долл. интерпретируется как гипотетическая цена нового автомобиля с нулевым пробегом (экстраполяция на нулевой пробег требует осторожности). Коэффициент $w_2 = -0.2$ означает, что каждая дополнительная тысяча километров пробега снижает цену в среднем на 200 долларов. На столь малом числе наблюдений модель, естественно, даёт идеальное описание, однако для статистически обоснованных выводов требуется больший объём данных.

---

## Вопросы для самопроверки

### Для начинающих

1. Запишите модель множественной линейной регрессии и поясните смысл каждого параметра.
2. Что такое базисные функции и в каких ситуациях их применение оправдано? Приведите два примера.
3. Почему максимизация функции правдоподобия при нормальных ошибках приводит к методу наименьших квадратов?
4. Выпишите выражение для градиента функции потерь $E(\mathbf{w})$ и сформулируйте условие минимума в матричной форме.
5. Что такое мультиколлинеарность и почему она препятствует обращению матрицы $\Phi^T \Phi$?
6. Как число обусловленности матрицы $\Phi$ связано с устойчивостью решения МНК?
7. В рассмотренном числовом примере коэффициент $w_1$ оказался равным нулю. Означает ли это, что возраст автомобиля гарантированно не влияет на цену?
8. Каковы размерности матриц $\Phi^T \Phi$ и $\Phi^T \mathbf{t}$ при $M$ базисных функциях и $n$ наблюдениях?

### Для профессионалов

1. Проведите полный вывод градиента $E(\mathbf{w})$ от частной производной по одному весу до матричной формы $\Phi^T(\Phi\mathbf{w} - \mathbf{t})$.
2. Объясните связь между числом обусловленности $\kappa(\Phi)$ и устойчивостью решения нормальных уравнений. Почему при $\kappa \gg 1$ прямое обращение $\Phi^T \Phi$ ненадёжно?
3. Сравните решение МНК с помощью нормальных уравнений, QR-разложения и SVD по вычислительной сложности и устойчивости. В каких случаях SVD оказывается предпочтительнее?
4. Выведите несмещённую оценку дисперсии ошибки $s^2 = SSE/(n-M)$. Почему теряется именно $M$ степеней свободы?
5. Что делать, если матрица $\Phi^T \Phi$ вырождена? Опишите, как SVD позволяет получить решение, и укажите связь с псевдообратной матрицей.
6. Предложите альтернативный вывод нормальных уравнений через транспонированную форму градиента и продемонстрируйте его эквивалентность основному подходу.

# Глава 3. Множественная линейная регрессия (часть II)

В предыдущих разделах мы получили аналитическое решение задачи наименьших квадратов $\hat{\mathbf{w}} = (\Phi^T \Phi)^{-1} \Phi^T \mathbf{t}$. Это решение изящно и точно для задач умеренной размерности. Однако с ростом объёма данных $n$ и особенно числа признаков $M$ прямое обращение матрицы $\Phi^T \Phi$ становится вычислительно непосильным, а иногда и численно неустойчивым. В этой части главы мы обратимся к итеративным методам оптимизации, не требующим обращения матриц и позволяющим эффективно находить $\hat{\mathbf{w}}$ даже в масштабных задачах. Центральное место среди них занимает градиентный спуск и его стохастические варианты, ставшие фундаментом современного машинного обучения.

Исходный код примеров написан на Python; все необходимые библиотеки импортируются один раз в начале:

```python
import numpy as np
import matplotlib.pyplot as plt
import time
```


### 0. Геометрический смысл градиента

Прежде чем переходить к итеративным методам, напомним, почему движение против градиента приводит нас к минимуму. Пусть $f(\mathbf{w})$ — дифференцируемая функция многих переменных. Градиент $\nabla f(\mathbf{w})$ — это вектор, составленный из частных производных:

$$
\nabla f(\mathbf{w}) = \begin{pmatrix}
\frac{\partial f}{\partial w_0} \\
\frac{\partial f}{\partial w_1} \\
\vdots \\
\frac{\partial f}{\partial w_{M-1}}
\end{pmatrix}.
$$

Важнейшее свойство градиента: он указывает направление наискорейшего возрастания функции. Чтобы понять это, рассмотрим производную функции $f$ по направлению единичного вектора $\mathbf{u}$ (т.е. $\|\mathbf{u}\| = 1$). По определению,

$$
D_{\mathbf{u}} f(\mathbf{w}) = \lim_{h \to 0} \frac{f(\mathbf{w} + h\mathbf{u}) - f(\mathbf{w})}{h}.
$$

Можно показать (используя цепное правило), что $D_{\mathbf{u}} f(\mathbf{w}) = \nabla f(\mathbf{w})^T \mathbf{u}$. Поскольку скалярное произведение двух векторов равно произведению их длин на косинус угла $\theta$ между ними, получаем

$$
D_{\mathbf{u}} f(\mathbf{w}) = \|\nabla f(\mathbf{w})\| \cdot \|\mathbf{u}\| \cos\theta = \|\nabla f(\mathbf{w})\| \cos\theta.
$$

Это выражение достигает максимума при $\cos\theta = 1$, т.е. когда $\mathbf{u}$ совпадает по направлению с $\nabla f(\mathbf{w})$. Таким образом, градиент действительно указывает направление, в котором функция возрастает быстрее всего, а его длина равна скорости этого возрастания. Следовательно, двигаясь в противоположном направлении ($-\nabla f$), мы наиболее быстро убываем. Именно на этом основан градиентный спуск.

Другое важное свойство — ортогональность градиента линиям уровня. Если $S = \{\mathbf{w} : f(\mathbf{w}) = c\}$ — поверхность постоянного значения функции, то градиент в любой точке этой поверхности перпендикулярен касательной к ней. Действительно, для любого малого смещения $\boldsymbol{\varepsilon}$ вдоль поверхности (т.е. $f(\mathbf{w}+\boldsymbol{\varepsilon}) = f(\mathbf{w})$) из разложения Тейлора

$$
f(\mathbf{w}+\boldsymbol{\varepsilon}) \approx f(\mathbf{w}) + \nabla f(\mathbf{w})^T \boldsymbol{\varepsilon}
$$

следует $\nabla f(\mathbf{w})^T \boldsymbol{\varepsilon} = 0$, а это и означает ортогональность градиента и вектора смещения, касательного к линии уровня. Это свойство помогает наглядно представить процесс оптимизации: на каждом шаге градиентного спуска мы движемся перпендикулярно текущей линии уровня.

Теперь, вооружённые этим пониманием, перейдём к применению градиентного спуска в линейной регрессии.


## 1. Потребность в итеративных методах

Аналитическое решение $\hat{\mathbf{w}} = (\Phi^T \Phi)^{-1} \Phi^T \mathbf{t}$ требует формирования матрицы $\Phi^T \Phi$ размера $M \times M$ и её последующего обращения. Как мы уже обсуждали в разделе 7 предыдущей части, суммарная сложность этой операции составляет $O(M^2 n + M^3)$. Когда $n$ и $M$ достигают десятков и сотен тысяч, выполнить обращение матрицы за разумное время невозможно. Более того, накопление ошибок округления может сделать результат численно ненадёжным.

Итеративные методы, и прежде всего градиентный спуск, идут по иному пути. Они не формируют и не обращают матрицу $\Phi^T \Phi$ целиком, а последовательно улучшают вектор весов $\mathbf{w}$, используя на каждом шаге лишь исходные данные — матрицу $\Phi$ и вектор $\mathbf{t}$. Сложность одного шага градиентного спуска составляет $O(M n)$, а общее число итераций $k$ обычно значительно меньше $M$, что даёт итоговую сложность $O(k M n)$. Во многих практических ситуациях это оказывается значительно выгоднее.

Напомним выражение для градиента функции потерь $E(\mathbf{w}) = \frac{1}{2}\sum_{i=1}^{n} (t_i - \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i))^2$, выведенное ранее:

$$
\nabla E(\mathbf{w}) = \Phi^T (\Phi \mathbf{w} - \mathbf{t}).
$$

Именно этот градиент будет направлять наши итерации.

## 2. Пакетный градиентный спуск (Batch Gradient Descent)

Самый прямой способ использовать градиент для оптимизации — **пакетный градиентный спуск** (batch gradient descent). На каждом шаге мы вычисляем градиент функции потерь по всей обучающей выборке и делаем шаг в направлении, противоположном градиенту:

$$
\mathbf{w}^{(k+1)} = \mathbf{w}^{(k)} - \alpha \nabla E(\mathbf{w}^{(k)}).
$$

Здесь $\alpha > 0$ — **скорость обучения** (learning rate). Это гиперпараметр, контролирующий величину шага. Можно представить себе спуск с горы в густом тумане: вы не видите дна долины, но можете определить направление, в котором склон уходит вниз круче всего, и делаете небольшой шаг. Затем снова оцениваете уклон и повторяете процедуру.

В матричной форме правило обновления записывается как

$$
\mathbf{w}^{(k+1)} = \mathbf{w}^{(k)} - \alpha\, \Phi^T (\Phi \mathbf{w}^{(k)} - \mathbf{t}). \tag{2.1}
$$

**Псевдокод пакетного градиентного спуска:**

```
Инициализировать w (например, нулями)
Повторять до сходимости:
    g = Φᵀ (Φ w - t)
    w ← w - α * g
```

Выбор скорости обучения $\alpha$ критически важен. При слишком маленьком $\alpha$ спуск будет медленным и потребует множества итераций. При слишком большом $\alpha$ алгоритм может начать осциллировать или даже расходиться, перепрыгивая через минимум. Геометрически линии уровня функции $E(\mathbf{w})$ являются эллипсоидами. Если они сильно вытянуты (плохая обусловленность), градиентный спуск с постоянным $\alpha$ может «метаться» по узкому оврагу, а не двигаться прямо ко дну.

Проиллюстрируем это поведение на модельном двумерном примере. Сгенерируем небольшую задачу с двумя параметрами (для наглядности) и изобразим траектории градиентного спуска для разных скоростей обучения на фоне линий уровня функции потерь.

```python
# Генерируем простые данные (M=2, без свободного члена для двумерного графика)
np.random.seed(0)
n_vis = 200
X_vis = np.random.randn(n_vis, 2)
# Истинные коэффициенты и зашумлённая целевая переменная
w_true_vis = np.array([3.0, -2.0])
t_vis = X_vis @ w_true_vis + 0.5 * np.random.randn(n_vis)

# Функция потерь (MSE) и её градиент для двумерного случая
def loss_2d(w):
    residuals = X_vis @ w - t_vis
    return 0.5 * np.sum(residuals**2)

def grad_2d(w):
    residuals = X_vis @ w - t_vis
    return X_vis.T @ residuals

# Сетка для линий уровня
w1_range = np.linspace(-1, 7, 200)
w2_range = np.linspace(-6, 2, 200)
W1, W2 = np.meshgrid(w1_range, w2_range)
Z = np.array([[loss_2d(np.array([w1, w2])) for w1 in w1_range] for w2 in w2_range])

# Запуск градиентного спуска с разными alpha
alphas_vis = [0.001, 0.005, 0.02]
n_steps = 50
trajectories = {}
for alpha in alphas_vis:
    w = np.array([0.0, 0.0])  # начальная точка
    traj = [w.copy()]
    for _ in range(n_steps):
        w = w - alpha * grad_2d(w)
        traj.append(w.copy())
    trajectories[alpha] = np.array(traj)

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, alpha in zip(axes, alphas_vis):
    ax.contour(W1, W2, Z, levels=30, cmap='viridis', alpha=0.6)
    traj = trajectories[alpha]
    ax.plot(traj[:, 0], traj[:, 1], 'r.-', markersize=3, linewidth=1)
    ax.plot(traj[0, 0], traj[0, 1], 'bo', label='Старт')
    ax.plot(traj[-1, 0], traj[-1, 1], 'r*', markersize=10, label='Финиш')
    ax.set_title(f'α = {alpha}')
    ax.set_xlabel('w₁')
    ax.set_ylabel('w₂')
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle('Траектории пакетного градиентного спуска при разных α')
plt.tight_layout()
plt.show()
```

На графиках видно, что при малом $\alpha$ траектория плавно, но медленно приближается к центру эллипсоидов (минимуму). При увеличении $\alpha$ скорость возрастает, но при слишком большом шаге траектория начинает осциллировать, а в экстремальных случаях — расходится.

> **Углублённый взгляд: сходимость и число обусловленности.**  
> Для выпуклой дифференцируемой функции с липшицевым градиентом градиентный спуск с постоянным шагом $\alpha \le 1/L$ (где $L$ — константа Липшица градиента) гарантированно сходится к минимуму. Для квадратичной функции $E(\mathbf{w}) = \frac{1}{2}\mathbf{w}^T \Phi^T \Phi \mathbf{w} - \mathbf{w}^T \Phi^T \mathbf{t} + \text{const}$ матрица Гессе равна $\Phi^T \Phi$, а $L$ — её максимальное собственное значение. Скорость сходимости при этом определяется числом обусловленности $\kappa = \lambda_{\max}/\lambda_{\min}$: чем ближе $\kappa$ к единице, тем быстрее сходимость. При $\kappa \gg 1$ (овражный ландшафт) сходимость становится очень медленной; в этом случае используют ускоряющие техники (momentum, адаптивные методы).

## 3. Стохастический градиентный спуск (SGD) и Mini-batch

Пакетный градиентный спуск требует вычисления градиента по всем $n$ наблюдениям на каждом шаге. При $n$, исчисляемом миллионами, это слишком дорого. **Стохастический градиентный спуск** (Stochastic Gradient Descent, SGD) радикально решает эту проблему: на каждой итерации используется градиент, вычисленный лишь по одному случайно выбранному наблюдению.

**Псевдокод SGD:**

```
Инициализировать w
Повторять до сходимости:
    выбрать случайный индекс i из {1,…,n}
    g = (wᵀ φ(xᵢ) - tᵢ) φ(xᵢ)
    w ← w - α * g
```

Формально правило обновления:

$$
\mathbf{w}^{(k+1)} = \mathbf{w}^{(k)} - \alpha \bigl(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}_i) - t_i\bigr) \boldsymbol{\phi}(\mathbf{x}_i),
$$

где индекс $i$ выбирается случайно из $\{1,\dots,n\}$.

Преимущества SGD:
- **Скорость:** каждый шаг выполняется за $O(M)$ операций (вместо $O(Mn)$), что позволяет быстро обрабатывать огромные выборки.
- **Память:** не требуется хранить всю матрицу $\Phi$ в оперативной памяти; можно подгружать данные по одному объекту.
- **Стохастичность:** шум в оценке градиента помогает алгоритму «выпрыгивать» из неглубоких локальных минимумов в сложных невыпуклых задачах (что особенно важно в нейронных сетях).

Недостатки:
- Оценка градиента сильно зашумлена, что приводит к нестабильной траектории сходимости.
- Требуется тщательный подбор расписания скорости обучения $\alpha$.

**Mini-batch градиентный спуск** представляет собой компромисс между пакетным и стохастическим подходами. На каждом шаге случайным образом выбирается небольшое подмножество (батч) из $B$ наблюдений, и градиент вычисляется по этому батчу. Это уменьшает шум по сравнению с чистым SGD и в то же время остаётся значительно быстрее полного пакетного градиента.

**Псевдокод Mini-batch GD:**

```
Инициализировать w
Повторять до сходимости:
    выбрать случайный батч индексов I размера B
    g = (1/B) ∑_{i∈I} (wᵀ φ(xᵢ) - tᵢ) φ(xᵢ)
    w ← w - α * g
```

На практике размер батча $B$ выбирают равным 32, 64, 128 и т.д., балансируя между стабильностью и вычислительной эффективностью.

Для обеспечения сходимости SGD скорость обучения $\alpha$ обычно постепенно уменьшают по мере увеличения числа итераций. Распространённые расписания:
- **Step decay:** $\alpha$ уменьшается в фиксированное число раз через определённые интервалы эпох.
- **Exponential decay:** $\alpha_k = \alpha_0 \, e^{-\gamma k}$.
- **Cosine annealing:** $\alpha_k = \alpha_{\min} + \frac{1}{2}(\alpha_{\max} - \alpha_{\min})(1 + \cos(\pi k / K))$, где $K$ — общее число итераций.

> **Углублённый взгляд: теорема сходимости SGD (Роббинс–Монро).**  
> Для выпуклой функции при определённых условиях на убывание $\alpha_k$ SGD сходится к глобальному минимуму в среднеквадратичном смысле. Классические условия Роббинса–Монро:
> $$
> \sum_{k=1}^{\infty} \alpha_k = \infty, \qquad \sum_{k=1}^{\infty} \alpha_k^2 < \infty.
> $$
> Первое условие гарантирует, что суммарный шаг достаточно велик, чтобы достичь любой точки пространства (шаги не затухают слишком быстро). Второе условие обеспечивает затухание дисперсии шума. Например, убывание $\alpha_k \propto 1/k$ удовлетворяет этим условиям. В невыпуклых задачах SGD может сходиться к локальному минимуму или седловой точке; на практике его шум помогает избегать плохих локальных минимумов, что и обуславливает успех в обучении нейронных сетей.

## 4. Неградиентные методы (расширенный обзор)

Помимо градиентных методов, существуют и другие итеративные подходы к решению задач наименьших квадратов. Остановимся на двух наиболее важных.

**Метод сопряжённых градиентов** (Conjugate Gradient, CG) специально предназначен для квадратичных функций. Он строит последовательность сопряжённых направлений поиска, используя текущий градиент и предыдущее направление. Для положительно определённой матрицы $\Phi^T \Phi$ CG находит точное решение не более чем за $M$ итераций (без учёта ошибок округления), причём каждая итерация стоит $O(M n)$ и требует лишь произведения матрицы на вектор. Метод не формирует и не хранит матрицу $\Phi^T \Phi$ в явном виде, что делает его идеальным для разреженных задач.

**Квазиньютоновские методы**, в частности **L‑BFGS** (Limited-memory Broyden–Fletcher–Goldfarb–Shanno), аппроксимируют обратную матрицу Гессе, используя историю градиентов. В отличие от полного BFGS, хранящего плотную матрицу $M \times M$, L‑BFGS хранит лишь несколько векторов длины $M$ (обычно от 3 до 20), что даёт линейную сложность по памяти $O(M)$. Метод обладает сверхлинейной скоростью сходимости и часто является наилучшим выбором для задач среднего размера (до нескольких тысяч признаков), когда аналитическое решение ещё возможно, но уже затратно.

Рекомендации по выбору метода:
- Огромные данные ($n > 10^6$, $M$ — большое): SGD или его адаптивные варианты (Adam, RMSProp).
- Задачи среднего размера ($n$ до $10^5$, $M$ до $10^4$): L‑BFGS или метод сопряжённых градиентов.
- Малые задачи ($M < 1000$): аналитическое решение через нормальные уравнения (если матрица обусловлена).

## 5. Реализация SGD на Python с нуля

Перейдём к практической демонстрации. Сгенерируем искусственный набор данных с $n = 10\,000$ наблюдений и $M = 3$ признаками (плюс свободный член, итого $M=4$ параметра). Истинные коэффициенты зададим как $\mathbf{w}_{\text{true}} = (5, -2, 0.5, 1)^T$, добавим нормальный шум $\varepsilon \sim \mathcal{N}(0, 1)$.

```python
np.random.seed(42)
n = 10000
M = 3  # число «настоящих» признаков (без учёта свободного члена)
w_true = np.array([5.0, -2.0, 0.5, 1.0])  # w0, w1, w2, w3

# Генерируем матрицу признаков (n x M)
X_raw = np.random.randn(n, M)
# Добавляем столбец единиц для свободного члена
Phi = np.hstack([np.ones((n, 1)), X_raw])
# Генерируем целевой вектор
t = Phi @ w_true + np.random.randn(n)
```

Реализуем несколько вариантов градиентного спуска: пакетный (batch), стохастический (SGD) и mini-batch. Функции градиента MSE (с множителем $1/2$ для удобства):

```python
def gradient_batch(Phi, t, w):
    """Градиент функции E(w) = 0.5 * sum((t - Phi @ w)^2)"""
    return Phi.T @ (Phi @ w - t)

def gradient_single(phi_i, t_i, w):
    """Градиент для одного наблюдения"""
    return (w @ phi_i - t_i) * phi_i
```

Теперь напишем функции обучения. Каждая возвращает историю значений функции потерь на всей выборке (для визуализации) и итоговый вектор весов.

```python
def batch_gd(Phi, t, w_init, alpha, n_iter):
    w = w_init.copy()
    loss_history = np.zeros(n_iter)
    for k in range(n_iter):
        w -= alpha * gradient_batch(Phi, t, w)
        loss_history[k] = 0.5 * np.sum((t - Phi @ w) ** 2)
    return w, loss_history

def sgd(Phi, t, w_init, alpha, n_iter):
    w = w_init.copy()
    n = Phi.shape[0]
    loss_history = np.zeros(n_iter)
    for k in range(n_iter):
        i = np.random.randint(n)
        w -= alpha * gradient_single(Phi[i], t[i], w)
        loss_history[k] = 0.5 * np.sum((t - Phi @ w) ** 2)
    return w, loss_history

def mini_batch_gd(Phi, t, w_init, alpha, n_iter, batch_size=100):
    w = w_init.copy()
    n = Phi.shape[0]
    loss_history = np.zeros(n_iter)
    for k in range(n_iter):
        idx = np.random.choice(n, batch_size, replace=False)
        Phi_b = Phi[idx]
        t_b = t[idx]
        w -= alpha * gradient_batch(Phi_b, t_b, w)
        loss_history[k] = 0.5 * np.sum((t - Phi @ w) ** 2)
    return w, loss_history
```

Запустим обучение для нескольких скоростей $\alpha$ и сравним сходимость. В качестве начального приближения возьмём нулевой вектор.

```python
w_init = np.zeros(M + 1)
n_iter = 2000

alphas = [1e-5, 3e-5, 1e-4]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, alpha in zip(axes, alphas):
    w_batch, loss_b = batch_gd(Phi, t, w_init, alpha, n_iter)
    w_sgd, loss_s = sgd(Phi, t, w_init, alpha, n_iter)
    w_mb, loss_mb = mini_batch_gd(Phi, t, w_init, alpha, n_iter, batch_size=100)
    ax.plot(loss_b, label='Batch', alpha=0.8)
    ax.plot(loss_s, label='SGD', alpha=0.8)
    ax.plot(loss_mb, label='Mini-batch (B=100)', alpha=0.8)
    ax.set_title(f'α = {alpha}')
    ax.set_xlabel('Итерация')
    ax.legend()
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Функция потерь')
plt.tight_layout()
plt.show()
```

График покажет характерное поведение: пакетный градиентный спуск снижает потери плавно, SGD — сильно шумит, но в среднем движется к минимуму, mini-batch занимает промежуточную позицию. При правильно подобранной скорости обучения все три метода сходятся к оптимальным весам.

Сравним итоговые веса с аналитическим решением:

```python
w_analytic = np.linalg.solve(Phi.T @ Phi, Phi.T @ t)
print("Аналитическое решение:", w_analytic)
print("Batch GD (α=1e-4):  ", w_batch)
print("Mini-batch GD:       ", w_mb)
```

Коэффициенты будут близки, хотя SGD может давать небольшое отклонение из-за неполной сходимости при фиксированном числе итераций. Важно понимать, что на таком маленьком $M$ аналитическое решение находится практически мгновенно; истинное преимущество итеративных методов раскроется, когда $M$ и $n$ станут действительно большими.

## 6. Сравнение вычислительной эффективности

Проведём более масштабный эксперимент, чтобы увидеть, при каких размерах данных итеративные методы начинают выигрывать у точного аналитического решения. Зафиксируем $n = 10\,000$ и будем варьировать $M$ от 5 до 100, а затем зафиксируем $M = 20$ и будем менять $n$ от 1000 до 100 000. Для каждого сочетания замерим время (медиану по 5 запускам) аналитического решения (через `np.linalg.solve`), пакетного градиентного спуска (500 итераций) и mini-batch SGD (500 итераций, размер батча 100). Все методы инициализируются нулевыми весами.

```python
def measure_time(method, *args, n_runs=5):
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        method(*args)
        times.append(time.perf_counter() - start)
    return np.median(times)

# Эксперимент 1: фиксированное n, меняем M
n_fixed = 10000
M_range = [5, 10, 20, 50, 100]
times_analytic_M = []
times_batch_M = []
times_mb_M = []

for M_val in M_range:
    np.random.seed(123)
    X_raw = np.random.randn(n_fixed, M_val - 1)  # M_val включает свободный член
    Phi_M = np.hstack([np.ones((n_fixed, 1)), X_raw])
    w_true_M = np.random.randn(M_val)
    t_M = Phi_M @ w_true_M + 0.1 * np.random.randn(n_fixed)
    
    def analytic():
        np.linalg.solve(Phi_M.T @ Phi_M, Phi_M.T @ t_M)
    t_a = measure_time(analytic, n_runs=5)
    times_analytic_M.append(t_a)
    
    def batch_run():
        w = np.zeros(M_val)
        for _ in range(500):
            w -= 0.0001 * (Phi_M.T @ (Phi_M @ w - t_M))
    t_b = measure_time(batch_run, n_runs=5)
    times_batch_M.append(t_b)
    
    def mb_run():
        w = np.zeros(M_val)
        n = n_fixed
        for _ in range(500):
            idx = np.random.choice(n, 100, replace=False)
            Phi_b = Phi_M[idx]
            t_b = t_M[idx]
            w -= 0.0001 * (Phi_b.T @ (Phi_b @ w - t_b))
    t_mb = measure_time(mb_run, n_runs=5)
    times_mb_M.append(t_mb)
```

Аналогично проведём эксперимент с варьированием $n$ при $M=20$.

```python
M_fixed = 20
n_range = [1000, 5000, 10000, 50000, 100000]
times_analytic_n = []
times_batch_n = []
times_mb_n = []

for n_val in n_range:
    np.random.seed(123)
    X_raw = np.random.randn(n_val, M_fixed - 1)
    Phi_n = np.hstack([np.ones((n_val, 1)), X_raw])
    w_true_n = np.random.randn(M_fixed)
    t_n = Phi_n @ w_true_n + 0.1 * np.random.randn(n_val)
    
    def analytic_n():
        np.linalg.solve(Phi_n.T @ Phi_n, Phi_n.T @ t_n)
    times_analytic_n.append(measure_time(analytic_n))
    
    def batch_n():
        w = np.zeros(M_fixed)
        for _ in range(500):
            w -= 0.0001 * (Phi_n.T @ (Phi_n @ w - t_n))
    times_batch_n.append(measure_time(batch_n))
    
    def mb_n():
        w = np.zeros(M_fixed)
        n = n_val
        for _ in range(500):
            idx = np.random.choice(n, 100, replace=False)
            Phi_b = Phi_n[idx]
            t_b = t_n[idx]
            w -= 0.0001 * (Phi_b.T @ (Phi_b @ w - t_b))
    times_mb_n.append(measure_time(mb_n))
```

Построим графики зависимости времени выполнения от $M$ и от $n$.

```python
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(M_range, times_analytic_M, 'o-', label='Аналитическое')
ax1.plot(M_range, times_batch_M, 's--', label='Batch GD (500 ит.)')
ax1.plot(M_range, times_mb_M, '^--', label='Mini-batch GD (500 ит.)')
ax1.set_xlabel('Число признаков M')
ax1.set_ylabel('Время (сек)')
ax1.set_title('Время при n = 10 000')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(n_range, times_analytic_n, 'o-', label='Аналитическое')
ax2.plot(n_range, times_batch_n, 's--', label='Batch GD (500 ит.)')
ax2.plot(n_range, times_mb_n, '^--', label='Mini-batch GD (500 ит.)')
ax2.set_xlabel('Число наблюдений n')
ax2.set_ylabel('Время (сек)')
ax2.set_title('Время при M = 20')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
```

На левом графике видно, что при малых $M$ аналитическое решение почти мгновенно, но с ростом $M$ его время быстро увеличивается из-за кубической зависимости $O(M^3)$. Пакетный и mini-batch SGD растут медленнее, и при $M$ порядка нескольких сотен они начинают опережать аналитический метод. На правом графике аналитическое решение также растёт линейно по $n$, но с большим коэффициентом ($O(M^2 n)$), тогда как SGD имеет сложность $O(k M B)$, практически не зависящую от $n$ при фиксированном числе итераций (поскольку мы используем батч постоянного размера). При достаточно больших $n$ SGD оказывается быстрее.

Отдельно стоит отметить потребление памяти: аналитическое решение хранит матрицу $\Phi^T \Phi$ размера $M \times M$ (т.е. $O(M^2)$), в то время как SGD в любой момент времени оперирует лишь текущим вектором весов $\mathbf{w}$ ($O(M)$) и текущим батчем данных. Это делает SGD единственно возможным выбором, когда матрица $\Phi$ не помещается в оперативной памяти.

Таким образом, выбор между аналитическим и итеративным методами зависит от размеров задачи и доступных вычислительных ресурсов. Итеративные методы, и в особенности SGD, становятся незаменимыми в мире больших данных, где $n$ и $M$ измеряются миллионами.

---

## Вопросы для самопроверки

### Для начинающих

1. В чём принципиальное отличие стохастического градиентного спуска (SGD) от пакетного (batch) градиентного спуска?
2. Как влияет выбор скорости обучения $\alpha$ на поведение градиентного спуска? Что произойдёт при слишком маленьком и при слишком большом $\alpha$?
3. Что такое mini-batch градиентный спуск и зачем он нужен?
4. В каких ситуациях аналитическое решение МНК через обращение $\Phi^T \Phi$ становится неприменимым?
5. Почему в SGD обычно уменьшают скорость обучения по мере увеличения числа итераций?

### Для профессионалов

1. Объясните, как число обусловленности матрицы $\Phi$ влияет на скорость сходимости пакетного градиентного спуска.
2. В чём суть метода L‑BFGS и в каких задачах его применение предпочтительнее SGD?
3. Опишите основные стратегии расписания скорости обучения в SGD. Какая из них лучше подходит для сильно нестационарных задач?
4. Обсудите компромисс между смещением и дисперсией оценки градиента в mini-batch SGD. Как размер батча влияет на этот компромисс?
5. Почему SGD способен работать (и часто хорошо работает) для невыпуклых функций потерь, несмотря на отсутствие глобальных гарантий сходимости?



## Часть III. Подготовка признаков и мультиколлинеарность

В предыдущих разделах мы строили модель множественной линейной регрессии, полагая, что все признаки уже даны в удобной числовой форме и связаны с откликом линейно. Однако реальные данные редко бывают подготовлены идеально: в них встречаются категориальные переменные («цвет», «тип двигателя»), нелинейные зависимости, а между числовыми признаками может существовать сильная корреляция, разрушающая устойчивость оценок. Эта часть главы посвящена тому, как преобразовать исходные данные в форму, пригодную для классической линейной регрессии, и как обнаружить мультиколлинеарность, которая сводит на нет интерпретируемость модели.

Все необходимые для примеров библиотеки импортируются один раз в начале.

```python
import pandas as pd
import numpy as np
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.datasets import make_regression
```

### 1. Предположения классической множественной регрессии

Прежде чем заниматься подготовкой признаков, полезно ещё раз зафиксировать те предпосылки, на которых держится классическая линейная модель. Выполнение этих предположений обеспечивает корректность МНК-оценок, их эффективность и возможность применять стандартные статистические тесты.

1. **Линейность по параметрам.** Математическое ожидание отклика является линейной функцией параметров: $\mathbb{E}[t^{(i)}] = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}^{(i)})$. Заметим, что линейность требуется именно по $\mathbf{w}$, а не по исходным признакам — именно поэтому мы можем использовать базисные функции, сохраняя линейную структуру.

2. **Независимость ошибок.** Случайные ошибки $\varepsilon_i$ не коррелируют между собой: $\mathrm{Cov}(\varepsilon_i, \varepsilon_j) = 0$ для $i \ne j$. Нарушение этого условия часто возникает во временных рядах или при кластеризованных данных.

3. **Гомоскедастичность.** Дисперсия ошибок постоянна: $\mathrm{Var}(\varepsilon_i) = \sigma^2$ для всех $i$. Если разброс ошибок меняется с ростом предсказанных значений, говорят о гетероскедастичности.

4. **Нормальность ошибок** (необходима для точных статистических выводов). $\varepsilon_i \sim \mathcal{N}(0,\sigma^2)$. В сочетании с независимостью это даёт точное распределение оценок, t- и F-тестов на выборках любого размера. При больших выборках центральная предельная теорема позволяет обойтись без нормальности.

5. **Отсутствие совершенной мультиколлинеарности.** Столбцы матрицы плана $\Phi$ должны быть линейно независимы. Если один признак точно выражается через другие, матрица $\Phi^T \Phi$ вырождена и единственного МНК-решения не существует.

При выполнении условий 1–4 оценки МНК являются **наилучшими линейными несмещёнными оценками** (теорема Гаусса–Маркова, BLUE). Добавление нормальности ошибок позволяет строить точные доверительные интервалы и проверять гипотезы с помощью t- и F-статистик.

### 2. Подготовка признаков: категориальные переменные и взаимодействия

#### 2.1. Категориальные признаки и one-hot кодирование

В разделе о базисных функциях мы упомянули, что категориальный признак можно представить набором индикаторных функций. Теперь развернём эту идею практически. Предположим, мы хотим предсказать цену автомобиля, и среди признаков есть «цвет кузова»: красный, синий или чёрный. Модель (1.1) не может работать со строкой «красный» — ей нужны числа. Решение состоит в создании нескольких бинарных столбцов, каждый из которых отвечает за одну категорию.

Процесс называется **one-hot кодированием** (или прямым унитарным кодированием). Для каждого возможного значения категории мы создаём отдельный признак, принимающий значение 1, если объект принадлежит этой категории, и 0 в противном случае. Рассмотрим игрушечный датасет с питомцами:

| Тип питомца | Цвет   | Вес (кг) |
|-------------|--------|----------|
| Собака      | Чёрный | 25       |
| Кошка       | Белый  | 4        |
| Собака      | Белый  | 30       |
| Кролик      | Чёрный | 2        |

После one-hot кодирования по столбцам «Тип питомца» и «Цвет» получим таблицу (без удаления какого-либо уровня):

| Вес | Тип_Собака | Тип_Кошка | Тип_Кролик | Цвет_Белый | Цвет_Чёрный |
|-----|------------|-----------|------------|------------|-------------|
| 25  | 1          | 0         | 0          | 0          | 1           |
| 4   | 0          | 1         | 0          | 1          | 0           |
| 30  | 1          | 0         | 0          | 1          | 0           |
| 2   | 0          | 0         | 1          | 0          | 1           |

Каждый объект теперь описывается набором нулей и единиц, и модель $t = w_0 + \sum w_j \phi_j(\mathbf{x})$ может быть построена. Однако возникает проблема: сумма столбцов «Тип_Собака», «Тип_Кошка» и «Тип_Кролик» в каждой строке равна единице. Но в нашей матрице плана уже есть столбец из единиц для свободного члена $w_0$. Следовательно, столбец единиц линейно выражается через сумму трёх индикаторов, и наоборот — один из индикаторов можно выразить через единичный столбец минус два других. Это порождает **совершенную мультиколлинеарность**: матрица $\Phi^T \Phi$ оказывается вырожденной, и модель не может быть однозначно оценена.

#### 2.2. Ловушка dummy-переменных

Разберём этот феномен формально. Пусть категориальный признак имеет $K$ уровней, и мы создали $K$ индикаторных столбцов $d_1, d_2, \dots, d_K$, где $d_{j}^{(i)} = 1$, если наблюдение $i$ принадлежит категории $j$. Тогда для любого $i$ выполнено

$$
\sum_{j=1}^{K} d_j^{(i)} = 1.
$$

В линейной модели со свободным членом

$$
t^{(i)} = w_0 + w_1 d_1^{(i)} + \dots + w_K d_K^{(i)} + \varepsilon^{(i)},
$$

свободный член $w_0$ умножается на единичный столбец, который совпадает с суммой индикаторов. Отсюда следует, что столбцы матрицы плана линейно зависимы. Если мы попытаемся вычислить МНК-оценку, матрица $\Phi^T \Phi$ окажется вырожденной и решение не будет единственным. Любые коэффициенты, отличающиеся на константу, добавленную к $w_0$ и вычтенную из всех $w_j$, будут давать идентичные предсказания.

Решение простое и элегантное: достаточно исключить **один** из индикаторных столбцов, например первый. Тогда оставшиеся $K-1$ столбцов вместе с единичным столбцом образуют линейно независимый набор. Исключённая категория становится **базовым уровнем**: её эффект «впитывается» в свободный член. Например, если мы удалим индикатор «Тип_Собака», то коэффициент $w_0$ будет соответствовать средней цене для собак (при прочих равных), а коэффициенты при «Тип_Кошка» и «Тип_Кролик» покажут, на сколько цена для кошек или кроликов отличается от базового уровня.

Именно поэтому в библиотеках pandas и sklearn есть параметр `drop_first=True` (или `drop='first'`), автоматически отбрасывающий первый уровень.

Проиллюстрируем это на Python, используя уже подготовленный маленький DataFrame.

```python
# Создаём игрушечный датафрейм
data = pd.DataFrame({
    'pet_type': ['Собака', 'Кошка', 'Собака', 'Кролик'],
    'color': ['Чёрный', 'Белый', 'Белый', 'Чёрный'],
    'weight': [25, 4, 30, 2]
})
print("Исходные данные:")
print(data)

# One-hot кодирование без удаления столбца (ловушка)
data_full = pd.get_dummies(data, columns=['pet_type', 'color'], dtype=int)
print("\nOne-hot без drop_first:")
print(data_full.head())
print("Сумма столбцов pet_type:", data_full[['pet_type_Собака','pet_type_Кошка','pet_type_Кролик']].sum(axis=1).values)
```

Мы увидим, что сумма столбцов pet_type равна единице для каждого наблюдения.

Теперь применим корректное кодирование с `drop_first=True` и построим модель линейной регрессии, предсказывающую `weight` по закодированным признакам.

```python
# Правильное кодирование
data_correct = pd.get_dummies(data, columns=['pet_type', 'color'], drop_first=True, dtype=int)
print("\nOne-hot с drop_first=True:")
print(data_correct.head())

# Подготовка матрицы X (с единичным столбцом, который добавляется statsmodels автоматически)
X = data_correct.drop('weight', axis=1)
y = data_correct['weight']
X = sm.add_constant(X)  # добавляем intercept

model = sm.OLS(y, X).fit()
print("\nРезультаты OLS:")
print(model.summary())
```

В сводке `model.summary()` мы увидим разумные коэффициенты: свободный член соответствует базовому уровню (комбинации Собака и Чёрный, если они были первыми в алфавитном порядке и выпали), а остальные коэффициенты показывают отклонения от него. Модель оценена без проблем.

> **Углублённый взгляд.** Альтернативным подходом является удаление свободного члена и оставление всех $K$ индикаторов. Тогда каждый коэффициент напрямую интерпретируется как среднее значение отклика для соответствующей категории. Однако такой подход делает неудобным сравнение вложенных моделей и смешивает интерпретацию с другими числовыми признаками. Классическая практика — сохранять свободный член и удалять один уровень.

#### 2.3. Взаимодействия признаков

Линейная модель предполагает, что влияние каждого признака аддитивно и не зависит от значений других признаков. В реальности эффекты могут быть взаимосвязаны: например, влияние пробега на цену автомобиля может быть сильнее для старых автомобилей, чем для новых. Чтобы учесть такой синергический эффект, в модель вводят **взаимодействия** — произведения признаков.

Формально, если у нас есть два числовых признака $x_1$ и $x_2$, мы можем добавить новый признак $x_1 x_2$ как дополнительную базисную функцию. Тогда модель примет вид

$$
t = w_0 + w_1 x_1 + w_2 x_2 + w_{12} (x_1 x_2) + \varepsilon.
$$

Производная $\partial \hat{t} / \partial x_1$ теперь равна $w_1 + w_{12} x_2$, то есть влияние $x_1$ зависит от уровня $x_2$. Это позволяет модели улавливать неаддитивные эффекты.

На практике при добавлении взаимодействий полезно предварительно центрировать переменные (вычитать среднее). Это уменьшает мультиколлинеарность между главными эффектами и произведением, делая оценки более стабильными, хотя и не меняет предсказательную силу модели.

Демонстрация на примере автомобилей: добавим взаимодействие возраста и пробега.

```python
# Пример: данные о машинах (возраст, пробег, цена)
np.random.seed(0)
n = 50
age = np.random.uniform(1, 15, n)
mileage = np.random.uniform(10, 200, n)
# Истинная модель с взаимодействием
price = 20 - 0.8 * age - 0.05 * mileage + 0.02 * age * mileage + np.random.normal(0, 1.5, n)

data_cars = pd.DataFrame({'age': age, 'mileage': mileage, 'price': price})

# Модель без взаимодействия
X_base = sm.add_constant(data_cars[['age', 'mileage']])
model_base = sm.OLS(data_cars['price'], X_base).fit()
print("R² без взаимодействия:", model_base.rsquared)

# Модель с взаимодействием (без центрирования)
data_cars['age_mileage'] = data_cars['age'] * data_cars['mileage']
X_inter = sm.add_constant(data_cars[['age', 'mileage', 'age_mileage']])
model_inter = sm.OLS(data_cars['price'], X_inter).fit()
print("R² с взаимодействием:", model_inter.rsquared)
print(model_inter.summary())
```

Обычно модель с взаимодействием имеет более высокий $R^2$ и значимый коэффициент при произведении, если эффект действительно присутствует. При необходимости можно также добавить центрирование и сравнить VIF для улучшения стабильности.

### 3. Мультиколлинеарность

#### 3.1. Понятие и последствия

**Мультиколлинеарность** — это высокая линейная корреляция между двумя или несколькими признаками (или базисными функциями). В отличие от совершенной мультиколлинеарности, которая делает решение невозможным, просто высокая корреляция не нарушает теоретических свойств МНК-оценок, но серьёзно ухудшает их практическую пригодность.

Основные последствия мультиколлинеарности:
- **Раздутые стандартные ошибки** коэффициентов: оценки становятся неустойчивыми, их дисперсии $\mathrm{Var}(\hat{w}_j) = \sigma^2 [(\Phi^T \Phi)^{-1}]_{jj}$ резко возрастают.
- **Нестабильность знаков и значений**: малое изменение данных может привести к изменению знака коэффициента.
- **Ложная незначимость**: t-статистики коэффициентов могут оказаться малыми, хотя на самом деле признаки влияют на отклик (из-за большой дисперсии).

Иными словами, при мультиколлинеарности модель теряет свою интерпретируемость: мы не можем надёжно сказать, какой из скоррелированных признаков «важнее» и на сколько.

#### 3.2. Коэффициент инфляции дисперсии (VIF)

Для количественной оценки мультиколлинеарности используют **коэффициент инфляции дисперсии** (Variance Inflation Factor, VIF). Для каждого признака $x_j$ вычисляется вспомогательная регрессия этого признака на все остальные и определяется её коэффициент детерминации $R_j^2$. Тогда

$$
\text{VIF}_j = \frac{1}{1 - R_j^2}.
$$

Если признак почти линейно выражается через остальные, $R_j^2 \approx 1$ и VIF стремится к бесконечности. На практике пороговым значением часто считают VIF > 5 (умеренная мультиколлинеарность) или VIF > 10 (серьёзная проблема).

> **Углублённый взгляд: VIF и собственные значения корреляционной матрицы.**  
> Если мы работаем со стандартизованными признаками (нулевое среднее, единичная дисперсия), то матрица $\Phi^T \Phi$ с точностью до множителя $1/n$ совпадает с выборочной корреляционной матрицей $R$. Тогда VIF для $j$-го коэффициента есть $j$-й диагональный элемент обратной корреляционной матрицы:  
> $$
> \text{VIF}_j = (R^{-1})_{jj}.
> $$  
> Это представление позволяет связать мультиколлинеарность со спектром матрицы $R$. Если собственные значения $R$ обозначить $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_M > 0$, то наличие очень малых $\lambda_k$ приводит к большим элементам $R^{-1}$ и, как следствие, к большим VIF. В частности, число обусловленности корреляционной матрицы $\kappa(R) = \lambda_{\max}/\lambda_{\min}$ может служить общим индикатором мультиколлинеарности: значения $\kappa > 30$ уже считаются тревожным сигналом. Таким образом, VIF и собственные значения — два взгляда на одну и ту же проблему линейной зависимости признаков.

#### 3.3. Демонстрация на Python

Сгенерируем набор данных с мультиколлинеарностью с помощью `make_regression`, задав высокую корреляцию между частью признаков.

```python
np.random.seed(42)
X, y, coef = make_regression(n_samples=200, n_features=5, n_informative=3,
                             noise=10, coef=True, random_state=42)

# Искусственно усилим корреляцию: добавим признак, почти равный сумме двух других
X = np.hstack([X, X[:, 0:1] + 0.05 * np.random.randn(200, 1)])
feature_names = [f'x{i}' for i in range(X.shape[1])]

# DataFrame для удобства
df = pd.DataFrame(X, columns=feature_names)
df['y'] = y

# Тепловая карта корреляций
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Матрица корреляций признаков')
plt.show()
```

Вычислим VIF для каждого признака.

```python
# Вычисление VIF
def calculate_vif(df, target_col='y'):
    features = df.drop(columns=[target_col])
    vif_data = pd.DataFrame({
        'feature': features.columns,
        'VIF': [variance_inflation_factor(features.values, i)
                for i in range(features.shape[1])]
    })
    return vif_data

vif_df = calculate_vif(df)
print("\nКоэффициенты инфляции дисперсии (VIF):")
print(vif_df)
```

Признаки с VIF > 10 (а тем более > 100) чётко указывают на проблему мультиколлинеарности. На тепловой карте мы увидим ярко выраженные ячейки с корреляцией, близкой к 1 или −1.

#### 3.4. Методы борьбы с мультиколлинеарностью

1. **Удаление части признаков.** Если два признака несут почти одинаковую информацию, один из них можно просто исключить. Выбор можно делать на основе VIF, доменного знания или корреляционного анализа.

2. **Регуляризация.** Методы Ridge ($L_2$-регуляризация) и Lasso ($L_1$-регуляризация) добавляют штраф к величине коэффициентов, что позволяет работать даже с сильно коррелированными признаками и автоматически отбирать наиболее важные из них. Этим методам будет посвящён отдельный раздел.

3. **Метод главных компонент (PCA).** Исходные коррелированные признаки заменяются небольшим числом ортогональных главных компонент, которые являются линейными комбинациями исходных. Это полностью устраняет мультиколлинеарность, но ценой потери прямой интерпретируемости коэффициентов.

4. **Центрирование и шкалирование.** При использовании взаимодействий и полиномиальных признаков полезно предварительно центрировать переменные (вычесть среднее) и, возможно, масштабировать. Это уменьшает корреляцию между исходной переменной и её степенями, улучшая численную устойчивость.

Выбор метода зависит от целей: если важна интерпретируемость, лучше удалить признаки или применить Lasso; если же цель — чистое предсказание, можно использовать PCA или Ridge.

Таким образом, грамотная подготовка признаков — залог построения надёжной и интерпретируемой линейной модели. Категориальные переменные должны быть преобразованы с осторожностью (с исключением одного уровня), взаимодействия добавлены осмысленно, а мультиколлинеарность своевременно выявлена и устранена.

Мы продолжаем третью часть главы, посвящённую диагностике модели, проверке статистических гипотез и оценке качества. Теперь, когда признаки подготовлены и мультиколлинеарность выявлена, необходимо убедиться, что построенная модель удовлетворяет предположениям, на которых основан классический регрессионный анализ. В этом нам поможет анализ остатков, формальные тесты и метрики. Все иллюстрации будем проводить на сквозном примере — данных о стоимости жилья в Калифорнии.

Для работы с California Housing и дополнительными статистическими функциями добавим необходимые импорты (основные библиотеки — pandas, numpy, statsmodels, matplotlib, seaborn — уже импортированы в предыдущей части):

```python
from sklearn.datasets import fetch_california_housing
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import shapiro
```

### 1. Анализ остатков

Остатки модели $\mathbf{e} = \mathbf{t} - \hat{\mathbf{t}}$ — это наблюдаемая реализация ненаблюдаемых ошибок $\boldsymbol{\varepsilon}$. Если классические предположения выполняются, то остатки должны выглядеть как белый шум: иметь нулевое среднее, постоянную дисперсию, не зависеть друг от друга и (для малых выборок) подчиняться нормальному распределению. Отклонения от этой картины подсказывают, какое именно предположение нарушено, и помогают выбрать способ коррекции модели.

В качестве сквозного примера возьмём набор данных California Housing, доступный в scikit-learn. Он содержит информацию о жилых кварталах Калифорнии: медианный доход жителей (`MedInc`), возраст домов (`HouseAge`), среднее количество комнат (`AveRooms`) и спален (`AveBedrms`), население (`Population`), среднюю занятость (`AveOccup`), широту (`Latitude`) и долготу (`Longitude`). Целевая переменная — медианная стоимость дома (`MedHouseVal`) в сотнях тысяч долларов. Загрузим данные и построим линейную модель со всеми признаками.

```python
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

# Добавляем константу и строим модель
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())
```

Получим предсказанные значения и остатки для диагностических графиков.

```python
fitted = model.fittedvalues
residuals = model.resid
```

Теперь построим четыре основных диагностических графика.

**a) Residuals vs Fitted.**  
По оси абсцисс откладываются предсказанные значения $\hat{t}_i$, по оси ординат — остатки $e_i$. Если модель адекватна, точки должны хаотично рассеиваться вокруг горизонтальной линии $y = 0$ без выраженной структуры. Наличие изгиба (U-образной или ∩-образной формы) указывает на нелинейность связи. Веерообразное расширение или сужение облака точек говорит о гетероскедастичности. Добавим сглаживающую LOWESS-линию с параметром `frac=0.6`, чтобы визуализировать возможный тренд.

```python
plt.figure(figsize=(8,6))
plt.scatter(fitted, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--', linewidth=1.5)
# LOWESS-сглаживание
lowess_line = lowess(residuals, fitted, frac=0.6)
plt.plot(lowess_line[:,0], lowess_line[:,1], color='green', linewidth=2)
plt.xlabel('Предсказанные значения')
plt.ylabel('Остатки')
plt.title('Residuals vs Fitted')
plt.grid(alpha=0.3)
plt.show()
```

На графике заметен лёгкий U-образный изгиб и расширение остатков с ростом предсказанных значений, что сигнализирует о возможной нелинейности и гетероскедастичности.

**b) Residuals vs каждый признак.**  
Построим диаграммы рассеяния остатков против каждого предиктора. Это помогает обнаружить признаки, по которым систематически меняется дисперсия или появляется нелинейный паттерн.

```python
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(X.columns):
    axes[i].scatter(X[col], residuals, alpha=0.3)
    axes[i].axhline(0, color='red', linestyle='--')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Остатки')
    axes[i].grid(alpha=0.3)
plt.suptitle('Остатки против признаков')
plt.tight_layout()
plt.show()
```

Наиболее выраженные паттерны часто видны для `AveRooms`, `AveBedrms` и `Population`, где есть выбросы и непостоянство дисперсии.

**c) Q‑Q plot.**  
Квантиль-квантильный график сравнивает выборочные квантили остатков с теоретическими квантилями нормального распределения. При нормальности точки должны лежать близко к диагональной прямой. Отклонения в хвостах указывают на более тяжёлые или лёгкие хвосты.

```python
fig = sm.qqplot(residuals, line='s')
plt.title('Q-Q plot остатков')
plt.grid(alpha=0.3)
plt.show()
```

Остатки демонстрируют заметное отклонение от нормальности на краях распределения, что неудивительно для данных о ценах с выбросами.

**d) Scale‑Location график.**  
Этот график предназначен для проверки гомоскедастичности. По вертикали откладывается $\sqrt{|r_i|}$, где $r_i$ — стьюдентизированные остатки (получаемые делением остатка на его стандартную ошибку), по горизонтали — предсказанные значения. Сглаживающая LOWESS-линия помогает увидеть тренд: если она горизонтальна, дисперсия постоянна; подъём или убывание указывает на гетероскедастичность.

```python
influence = model.get_influence()
stud_res = influence.resid_studentized_internal
sqrt_abs_resid = np.sqrt(np.abs(stud_res))

plt.figure(figsize=(8,6))
plt.scatter(fitted, sqrt_abs_resid, alpha=0.5)
lowess_line = lowess(sqrt_abs_resid, fitted, frac=0.6)
plt.plot(lowess_line[:,0], lowess_line[:,1], color='red', linewidth=2)
plt.xlabel('Предсказанные значения')
plt.ylabel('√|Стьюдентизированные остатки|')
plt.title('Scale-Location')
plt.grid(alpha=0.3)
plt.show()
```

На графике виден отчётливый восходящий тренд — дисперсия ошибок растёт с ростом предсказанной стоимости. Это подтверждает подозрение о гетероскедастичности.

### 2. Статистические тесты

Визуальную диагностику дополним формальными тестами.

**Тест Бройша–Пагана** проверяет нулевую гипотезу о гомоскедастичности. Вспомогательная регрессия квадратов остатков на исходные регрессоры даёт статистику $LM = n \cdot R^2_{\text{aux}}$, которая асимптотически распределена как $\chi^2$ с числом степеней свободы, равным количеству признаков.

```python
bp_test = het_breuschpagan(residuals, model.model.exog)
print(f"Тест Бройша–Пагана: LM={bp_test[0]:.2f}, p-value={bp_test[1]:.4f}")
```

**Тест Уайта** обобщает тест Бройша–Пагана, включая во вспомогательную регрессию квадраты и перекрёстные произведения признаков, что позволяет обнаружить более сложные формы гетероскедастичности.

```python
white_test = het_white(residuals, model.model.exog)
print(f"Тест Уайта: LM={white_test[0]:.2f}, p-value={white_test[1]:.4f}")
```

**Тест Дарбина–Уотсона** проверяет наличие автокорреляции первого порядка в остатках. Статистика $DW$ лежит между 0 и 4; значения около 2 свидетельствуют об отсутствии автокорреляции. Для пространственных данных, таких как California Housing, этот тест не совсем применим, но мы приведём его для иллюстрации, сделав соответствующую оговорку.

```python
dw = durbin_watson(residuals)
print(f"Статистика Дарбина–Уотсона: {dw:.3f} (значение около 2 указывает на отсутствие автокорреляции)")
```

**Тест Шапиро–Уилка** проверяет нормальность остатков. Нулевая гипотеза — данные взяты из нормального распределения.

```python
shapiro_stat, shapiro_p = shapiro(residuals)
print(f"Тест Шапиро–Уилка: W={shapiro_stat:.4f}, p-value={shapiro_p:.4f}")
```

**F‑тест общей значимости** проверяет гипотезу $H_0: w_1 = w_2 = \dots = w_D = 0$ против альтернативы, что хотя бы один коэффициент не равен нулю. Низкое p‑значение говорит о том, что модель значимо лучше константной.

```python
print(f"F-тест: F={model.fvalue:.2f}, p-value={model.f_pvalue:.4f}")
```

Результаты тестов (Бройш–Пагана и Уайта) дают p‑value значительно меньше 0.05, что отвергает гипотезу о гомоскедастичности. Тест Шапиро–Уилка также отвергает нормальность остатков (p‑value < 0.05). F‑тест подтверждает общую значимость модели.

### 3. Метрики качества

Для оценки предсказательной силы и сравнения моделей используют несколько метрик:

- $R^2$ — доля объяснённой дисперсии.
- Скорректированный $R^2_{\text{adj}} = 1 - \frac{SSE/(n-M)}{SST/(n-1)}$ — штрафует за добавление неинформативных признаков.
- MSE = $\frac{1}{n} \sum e_i^2$ и RMSE = $\sqrt{\text{MSE}}$ — средняя квадратичная ошибка и её корень, возвращающий ошибку в исходные единицы.
- Информационные критерии: AIC = $n \log(SSE/n) + 2M$, BIC = $n \log(SSE/n) + M \log n$ — чем меньше, тем лучше; BIC сильнее штрафует за сложность модели и позволяет сравнивать невложенные модели.

Вычислим их для текущей модели.

```python
n = len(y)
M = len(model.params)
sse = np.sum(residuals**2)
sst = np.sum((y - y.mean())**2)
r2 = 1 - sse/sst
r2_adj = 1 - (sse/(n-M)) / (sst/(n-1))
mse = sse / n
rmse = np.sqrt(mse)
aic = n * np.log(sse/n) + 2*M
bic = n * np.log(sse/n) + M * np.log(n)
print(f"R² = {r2:.4f}")
print(f"R²_adj = {r2_adj:.4f}")
print(f"MSE = {mse:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"AIC = {aic:.2f}")
print(f"BIC = {bic:.2f}")
```

### 4. Влиятельные наблюдения

Некоторые точки могут непропорционально сильно влиять на оценки коэффициентов. Для их выявления используют **расстояние Кука** $D_i$ — мера изменения вектора коэффициентов при удалении $i$-го наблюдения. Пороговым значением считают $4/n$. Также смотрят на **леверидж** $h_{ii}$ — диагональный элемент матрицы проекции $P = \Phi(\Phi^T \Phi)^{-1}\Phi^T$; большие значения говорят о том, что точка находится далеко от центра облака предикторов.

Построим столбчатую диаграмму расстояний Кука и отметим порог.

```python
cooks = model.get_influence().cooks_distance[0]
threshold = 4 / n
plt.figure(figsize=(12,4))
plt.bar(range(n), cooks, width=0.8)
plt.axhline(y=threshold, color='red', linestyle='--', label=f'Порог 4/n = {threshold:.4f}')
plt.xlabel('Номер наблюдения')
plt.ylabel('Расстояние Кука')
plt.title('Расстояние Кука для каждого наблюдения')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f"Число точек с D > 4/n: {np.sum(cooks > threshold)}")
```

Точки, превышающие порог, могут быть кандидатами на удаление или дополнительный анализ.

### 5. Сквозной пример диагностики и улучшение модели

Проведём полную диагностику текущей модели. Кроме уже полученных результатов, вычислим VIF для признаков, как делали в предыдущей части (можно использовать ту же функцию `calculate_vif`). Напомним, что высокий VIF указывает на мультиколлинеарность.

```python
# VIF (используем функцию calculate_vif, определённую ранее)
vif_df = calculate_vif(pd.concat([X, y.rename('MedHouseVal')], axis=1), target_col='MedHouseVal')
print("\nVIF для исходных признаков:")
print(vif_df)
```

Видим, что многие признаки имеют умеренный VIF, но некоторые (например, `AveRooms` и `AveBedrms`) коррелированы.

Теперь предложим улучшения:
1. **Логарифмирование целевой переменной.** Часто цены имеют логнормальное распределение; логарифмирование помогает стабилизировать дисперсию и приблизить распределение ошибок к нормальному.
2. **Добавление квадратичных членов** для широты и долготы, чтобы уловить нелинейное географическое влияние.
3. **Удаление влиятельных наблюдений** по расстоянию Кука.

Реализуем эти шаги.

```python
# Улучшенная модель
# Логарифмируем целевую переменную
y_log = np.log(y)
# Добавляем квадраты широты и долготы
X_improved = X.copy()
X_improved['Latitude_sq'] = X_improved['Latitude']**2
X_improved['Longitude_sq'] = X_improved['Longitude']**2
X_improved_const = sm.add_constant(X_improved)

# Удаляем выбросы по расстоянию Кука, полученные из предыдущей модели
# (чтобы не было утечки данных, пересчитаем выбросы после построения новой модели)
model_improved = sm.OLS(y_log, X_improved_const).fit()
cooks_improved = model_improved.get_influence().cooks_distance[0]
keep_idx = cooks_improved < (4 / len(y_log))
X_final = X_improved_const[keep_idx]
y_final = y_log[keep_idx]
model_final = sm.OLS(y_final, X_final).fit()
print(model_final.summary())
```

Сравним метрики исходной и финальной моделей. Для сопоставимости рассчитаем метрики финальной модели на тех же точках, которые использовались при обучении (можно также использовать кросс-валидацию, но для иллюстрации достаточно within-sample).

```python
# Метрики финальной модели
resid_final = model_final.resid
n_final = len(y_final)
M_final = len(model_final.params)
sse_final = np.sum(resid_final**2)
sst_final = np.sum((y_final - y_final.mean())**2)
r2_final = 1 - sse_final/sst_final
r2_adj_final = 1 - (sse_final/(n_final-M_final)) / (sst_final/(n_final-1))
mse_final = sse_final / n_final
rmse_final = np.sqrt(mse_final)
aic_final = n_final * np.log(sse_final/n_final) + 2*M_final
bic_final = n_final * np.log(sse_final/n_final) + M_final * np.log(n_final)

print("\nСравнение моделей (в логарифмической шкале):")
print(f"Исходная модель: R²={r2:.4f}, R²_adj={r2_adj:.4f}, AIC={aic:.2f}, BIC={bic:.2f}")
print(f"Финальная модель: R²={r2_final:.4f}, R²_adj={r2_adj_final:.4f}, AIC={aic_final:.2f}, BIC={bic_final:.2f}")
```

Финальная модель (с логарифмированием отклика, дополнительными квадратичными членами и удалением выбросов) обычно показывает улучшенные значения информационных критериев, а анализ остатков демонстрирует более равномерный разброс.

> **Углублённый взгляд: выбор между AIC и BIC.**  
> AIC и BIC оба оценивают качество модели, штрафуя за сложность, но с разной философией. AIC стремится выбрать модель, которая лучше всего предсказывает новые данные, и асимптотически эквивалентен минимизации ожидаемой информации (или максимизации ожидаемой логарифмической правдоподобности). BIC, напротив, имеет байесовское происхождение и при больших $n$ стремится выбрать истинную модель (если она есть среди рассматриваемых). На практике BIC сильнее штрафует за число параметров, поэтому склонен выбирать более экономные модели. При больших выборках (как в California Housing) разница между критериями может быть существенной: если AIC и BIC указывают на разные модели, предпочтение часто отдают модели, выбранной BIC, если цель — интерпретация и отбор значимых признаков.

Теперь мы имеем полный инструментарий для диагностики и улучшения линейной регрессии. Анализ остатков, формальные тесты и метрики качества позволяют объективно оценить адекватность модели и направить усилия на её доработку.

---

## Вопросы для самопроверки (части IIIA и IIIB)

### Для начинающих

1. Что такое VIF и для чего он используется? Как интерпретировать значения VIF больше 5 или 10?
2. Как выглядит гетероскедастичность на графике «Residuals vs Fitted» и на графике «Scale‑Location»?
3. Зачем нужен скорректированный коэффициент детерминации $R^2_{\text{adj}}$? В чём его отличие от обычного $R^2$?
4. Как правильно закодировать категориальный признак с тремя уровнями для использования в линейной регрессии? Что произойдёт, если не удалить один уровень?
5. Что показывает F-тест в выводе `model.summary()`? Как интерпретировать его p‑значение?

### Для профессионалов

1. Почему one-hot кодирование без удаления одного уровня приводит к совершенной мультиколлинеарности? Приведите аналитическое обоснование.
2. Какие альтернативы удалению одного уровня существуют при кодировании категориальных признаков? В каких ситуациях они уместны?
3. Что позволяет обнаружить Scale‑Location график, чего нельзя увидеть на обычном графике остатков против предсказанных значений?
4. Как связан VIF с собственными значениями корреляционной матрицы признаков? Почему при наличии малых собственных значений VIF становится большим?
5. Объясните разницу между AIC и BIC. Какой критерий предпочтительнее использовать при отборе модели для прогнозирования, а какой — для выбора «истинной» модели?

Мы существенно расширим раздел о регуляризации, добавив подробные выводы всех ключевых формул, градиентов и условий оптимальности. Это позволит читателю не только понять идею, но и воспроизвести вычисления.

---

## Регуляризация в линейной регрессии

В предыдущих разделах мы видели, что обычный метод наименьших квадратов даёт оптимальные оценки в смысле BLUE, но при большом числе признаков или их сильной корреляции оценки становятся неустойчивыми, а предсказательная способность модели падает из‑за переобучения. Регуляризация предлагает изменить функцию потерь, добавив штраф за величину коэффициентов, что делает решение более стабильным и часто улучшает обобщающую способность.

Импорт библиотек для практической части:

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV, lasso_path
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
```

### 1. Зачем нужна регуляризация?

Классический МНК минимизирует сумму квадратов ошибок

$$
E(\mathbf{w}) = \frac{1}{2}\|\mathbf{t} - \Phi\mathbf{w}\|^2.
$$

Если матрица $\Phi^T\Phi$ плохо обусловлена (например, из‑за мультиколлинеарности), её обращение даёт большие по модулю и нестабильные значения $\hat{\mathbf{w}}$. Когда $M > n$, матрица $\Phi^T\Phi$ вообще вырождена. Регуляризация добавляет к $E(\mathbf{w})$ штрафной член $R(\mathbf{w})$, который «стягивает» коэффициенты к нулю, делая решение более гладким:

$$
E_{\text{reg}}(\mathbf{w}) = \frac{1}{2}\|\mathbf{t} - \Phi\mathbf{w}\|^2 + \lambda R(\mathbf{w}),
$$

где $\lambda \ge 0$ управляет силой регуляризации. Вид $R(\mathbf{w})$ определяет свойства оценок.

### 2. Ridge регрессия ($L_2$-регуляризация)

В гребневой регрессии штрафуются квадраты коэффициентов (кроме свободного члена $w_0$):

$$
E_{\text{ridge}}(\mathbf{w}) = \frac{1}{2}\sum_{i=1}^{n}\bigl(t_i - \mathbf{w}^T\boldsymbol{\phi}(\mathbf{x}_i)\bigr)^2 + \frac{\lambda}{2}\sum_{j=1}^{M-1} w_j^2.
$$

Свободный член $w_0$ не включается в штраф, чтобы модель могла компенсировать средний уровень отклика. В матричной форме

$$
E_{\text{ridge}}(\mathbf{w}) = \frac{1}{2}\|\mathbf{t} - \Phi\mathbf{w}\|^2 + \frac{\lambda}{2}\mathbf{w}^T I' \mathbf{w},
$$

где $I'$ — диагональная матрица, у которой первый элемент (для $w_0$) равен 0, а остальные диагональные элементы равны 1. Иными словами, $I' = \operatorname{diag}(0,1,1,\dots,1)$.

Найдём градиент $E_{\text{ridge}}$ по $\mathbf{w}$. Градиент первого слагаемого мы уже вычисляли: $\nabla_{\mathbf{w}} \frac{1}{2}\|\mathbf{t} - \Phi\mathbf{w}\|^2 = \Phi^T(\Phi\mathbf{w} - \mathbf{t})$. Градиент штрафного члена равен $\lambda I' \mathbf{w}$. Таким образом,

$$
\nabla_{\mathbf{w}} E_{\text{ridge}} = \Phi^T(\Phi\mathbf{w} - \mathbf{t}) + \lambda I' \mathbf{w}.
$$

Приравнивая градиент к нулю, получаем нормальные уравнения с регуляризацией:

$$
\Phi^T\Phi\mathbf{w} + \lambda I' \mathbf{w} = \Phi^T\mathbf{t}.
$$

Объединяя матрицы, имеем $(\Phi^T\Phi + \lambda I')\mathbf{w} = \Phi^T\mathbf{t}$. Поскольку добавка $\lambda I'$ положительно полуопределена и делает матрицу лучше обусловленной, можно обратить:

$$
\boxed{\hat{\mathbf{w}}_{\text{ridge}} = (\Phi^T\Phi + \lambda I')^{-1}\Phi^T\mathbf{t}}.
$$

При $\lambda \to 0$ решение стремится к обычному МНК, а при $\lambda \to \infty$ все $w_j$ ($j\ge 1$) стремятся к нулю, и $w_0 \to \bar{t}$ (среднее отклика). Оптимальное $\lambda$ подбирается кросс‑валидацией.

> **Углублённый взгляд: байесовская интерпретация Ridge.**  
> Минимизация $E_{\text{ridge}}$ эквивалентна максимизации апостериорной вероятности в байесовской модели с нормальным априорным распределением коэффициентов: $w_j \sim \mathcal{N}(0,\tau^2)$ для $j=1,\dots,M-1$. Тогда логарифм апостериорной плотности пропорционален $-\frac{1}{2\sigma^2}\|\mathbf{t} - \Phi\mathbf{w}\|^2 - \frac{1}{2\tau^2}\sum_{j=1}^{M-1} w_j^2$. Умножая на $\sigma^2$ и сравнивая с $E_{\text{ridge}}$, видим, что $\lambda = \sigma^2/\tau^2$. Чем меньше $\tau^2$ (сильнее априорная уверенность в близости коэффициентов к нулю), тем больше $\lambda$.

### 3. Lasso регрессия ($L_1$-регуляризация)

Лассо (Least Absolute Shrinkage and Selection Operator) использует штраф в виде суммы абсолютных значений коэффициентов:

$$
E_{\text{lasso}}(\mathbf{w}) = \frac{1}{2}\|\mathbf{t} - \Phi\mathbf{w}\|^2 + \lambda \sum_{j=1}^{M-1} |w_j|.
$$

Главная особенность — способность обнулять часть коэффициентов, выполняя автоматический отбор признаков.

В отличие от Ridge, здесь нет аналитического решения, потому что функция $|w_j|$ не дифференцируема в нуле. Однако можно выписать условия оптимальности, используя понятие **субградиента**. Для выпуклой функции $f$ точка $\mathbf{w}^*$ является минимумом тогда и только тогда, когда нулевой вектор принадлежит субдифференциалу: $\mathbf{0} \in \partial f(\mathbf{w}^*)$.

Для Lasso субградиент по $w_j$ ($j \ge 1$) имеет вид:
$$
\frac{\partial E_{\text{lasso}}}{\partial w_j} =
\begin{cases}
-\bigl[\Phi^T(\mathbf{t} - \Phi\mathbf{w})\bigr]_j + \lambda, & w_j > 0,\\
-\bigl[\Phi^T(\mathbf{t} - \Phi\mathbf{w})\bigr]_j - \lambda, & w_j < 0,\\
\bigl[-\Phi^T(\mathbf{t} - \Phi\mathbf{w})\bigr]_j + \lambda \cdot [-1, 1], & w_j = 0.
\end{cases}
$$

Условие $\mathbf{0} \in \partial E_{\text{lasso}}$ приводит к системе:
$$
[\Phi^T(\mathbf{t} - \Phi\mathbf{w})]_j = \lambda \cdot \text{sign}(w_j), \quad \text{если } w_j \neq 0,
$$
$$
\bigl|[\Phi^T(\mathbf{t} - \Phi\mathbf{w})]_j\bigr| \le \lambda, \quad \text{если } w_j = 0.
$$

Отсюда видно, что если абсолютная величина «градиента» SSE по $w_j$ меньше $\lambda$, то коэффициент зануляется. Это и есть механизм отбора признаков. Чем больше $\lambda$, тем больше коэффициентов обнуляется.

Для решения задачи Lasso обычно применяют **координатный спуск** — итеративно обновляют один коэффициент $w_j$, фиксируя остальные, решая одномерную задачу минимизации, которая имеет явное решение через мягкое пороговое преобразование (soft‑thresholding):

$$
w_j^{\text{new}} = \mathcal{S}_{\lambda}\!\left(\frac{1}{\|\boldsymbol{\phi}_j\|^2}\bigl[\Phi^T(\mathbf{t} - \Phi\mathbf{w}_{-j})\bigr]_j\right),
$$
где $\mathcal{S}_{\lambda}(x) = \text{sign}(x)(|x| - \lambda)_+$ — оператор мягкого порога, а $\mathbf{w}_{-j}$ — вектор всех коэффициентов, кроме $j$-го.

> **Углублённый взгляд: байесовская интерпретация Lasso.**  
> Штраф $L_1$ соответствует априорному распределению Лапласа на коэффициенты: $p(w_j) \propto \exp(-\frac{|w_j|}{\tau})$. Острый пик плотности в нуле и относительно тяжёлые хвосты приводят к тому, что многие апостериорные оценки оказываются точно в нуле. Это и порождает разреженность.

### 4. Elastic Net

Эластичная сеть объединяет $L_1$- и $L_2$-штрафы:

$$
E_{\text{elastic}}(\mathbf{w}) = \frac{1}{2}\|\mathbf{t} - \Phi\mathbf{w}\|^2 + \lambda_1 \sum_{j=1}^{M-1} |w_j| + \frac{\lambda_2}{2} \sum_{j=1}^{M-1} w_j^2.
$$

Перепараметризуем в виде: $\alpha = \lambda_1 + \lambda_2$, $\rho = \lambda_1/(\lambda_1+\lambda_2)$ (в sklearn это `alpha` и `l1_ratio`). Тогда

$$
E_{\text{elastic}} = \frac{1}{2}\|\mathbf{t} - \Phi\mathbf{w}\|^2 + \alpha \rho \sum_{j=1}^{M-1} |w_j| + \alpha (1-\rho) \frac{1}{2} \sum_{j=1}^{M-1} w_j^2.
$$

Эта целевая функция дифференцируема всюду, кроме нулей, и может решаться модифицированным координатным спуском. Elastic Net сохраняет разреженность (благодаря $L_1$) и одновременно хорошо работает с группами сильно коррелированных признаков (благодаря $L_2$).

### 5. Реализация и сравнение на Python

Повторим практическую часть с пояснениями. Сгенерируем данные: 200 образцов, 50 признаков, из которых только 10 действительно влияют на отклик.

```python
np.random.seed(42)
X, y, true_coef = make_regression(n_samples=200, n_features=50, n_informative=10,
                                  noise=10, coef=True, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
```

Для выбора оптимальных гиперпараметров используем встроенные классы с кросс-валидацией. Сетка `alphas` задана в логарифмическом масштабе.

```python
alphas = np.logspace(-3, 3, 50)

ridge = RidgeCV(alphas=alphas, store_cv_values=True)
ridge.fit(X_train, y_train)

lasso = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000)
lasso.fit(X_train, y_train)

elastic = ElasticNetCV(alphas=alphas, l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
                       cv=5, random_state=42, max_iter=10000)
elastic.fit(X_train, y_train)
```

Посмотрим на коэффициенты и метрики на тесте:

```python
def evaluate(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    n_nonzero = np.sum(model.coef_ != 0)
    print(f"{name}: R²={r2:.3f}, MSE={mse:.2f}, ненулевых признаков={n_nonzero}")
    return r2, mse, n_nonzero

print("Оптимальные параметры:")
print(f"Ridge: λ={ridge.alpha_:.3f}")
print(f"Lasso: λ={lasso.alpha_:.3f}")
print(f"Elastic Net: λ={elastic.alpha_:.3f}, l1_ratio={elastic.l1_ratio_:.3f}")
evaluate(ridge, "Ridge", X_test, y_test)
evaluate(lasso, "Lasso", X_test, y_test)
evaluate(elastic, "Elastic Net", X_test, y_test)
```

Построим график регуляризационного пути для Lasso:

```python
alphas_path, coef_path, _ = lasso_path(X_train, y_train, alphas=alphas)
plt.figure(figsize=(10,6))
for i in range(coef_path.shape[0]):
    plt.semilogx(alphas_path, coef_path[i], lw=1)
plt.xlabel('λ')
plt.ylabel('Коэффициенты')
plt.title('Lasso regularization path')
plt.axvline(lasso.alpha_, color='red', ls='--', label=f'Выбранное λ={lasso.alpha_:.3f}')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```

Видно, как с ростом $\lambda$ коэффициенты сжимаются к нулю. Сравним отобранные номера признаков с истинными:

```python
true_support = np.where(true_coef != 0)[0]
lasso_support = np.where(lasso.coef_ != 0)[0]
print(f"Истинные номера: {true_support}")
print(f"Отобранные Lasso: {lasso_support}")
print(f"Пересечение: {np.intersect1d(true_support, lasso_support)}")
```

Лассо обычно находит большинство информативных признаков, но может пропустить слабые, если корреляция высока. Elastic Net в таких ситуациях часто работает лучше.

### 6. Практические рекомендации

- **Стандартизация.** Регуляризация налагает одинаковый штраф на все коэффициенты, поэтому признаки должны быть в одинаковом масштабе (нулевое среднее, единичная дисперсия). В sklearn `RidgeCV`, `LassoCV` и `ElasticNetCV` не делают этого автоматически — используйте `StandardScaler` в пайплайне.
- **Когда что применять.**  
  - **Ridge** — когда важны все признаки и нет необходимости в их отборе.  
  - **Lasso** — когда требуется автоматический отбор признаков и интерпретируемость.  
  - **Elastic Net** — компромисс при сильной корреляции признаков или когда Lasso выбирает слишком мало признаков.
- **Стандартные ошибки.** Обычная теория МНК не работает для регуляризованных оценок, так как они смещены. Для оценки значимости можно использовать **бутстреп** или методы дебайасинга (например, debiased Lasso), которые восстанавливают асимптотически нормальные и несмещённые оценки с корректными стандартными ошибками.

> **Углублённый взгляд: debiased Lasso.**  
> Идея состоит в том, чтобы после отбора модели с помощью Lasso скорректировать оценки коэффициентов, решая модифицированную систему уравнений, которая удаляет смещение, вносимое регуляризацией. Это позволяет строить доверительные интервалы и проверять гипотезы, сохраняя преимущества разреженности.

> **Углублённый взгляд: что означает линейная зависимость признаков?**  
> С точки зрения линейной алгебры, ранг произведения $\Phi^T\Phi$ совпадает с рангом $\Phi$. Поэтому $\Phi^T\Phi$ будет обратима ровно тогда, когда столбцы $\Phi$ линейно независимы. В контексте регрессии столбцы $\Phi$ — это признаки (возможно, после базисных преобразований). Их линейная зависимость означает, что один или несколько признаков могут быть выражены как линейные комбинации остальных. Например, если у нас есть признаки «общая площадь квартиры», «жилая площадь» и «площадь кухни», причём общая площадь всегда равна сумме жилой и кухни, то третий столбец матрицы $\Phi$ будет линейной комбинацией двух других, и $\Phi^T\Phi$ окажется вырожденной.  
>  
> На практике мы стараемся избегать такой ситуации, удаляя избыточные признаки ещё на этапе подготовки данных. Однако даже при отсутствии строгой линейной зависимости может возникать *приближённая* зависимость, когда один признак почти линейно выражается через другие. Тогда матрица $\Phi^T\Phi$ будет хотя и обратимой, но близкой к вырожденной, что приводит к серьёзным вычислительным проблемам — именно о них пойдёт речь в следующем разделе.


## Вопросы для самопроверки

### Для начинающих

1. Зачем нужен штрафной член в регуляризованной регрессии? Какую проблему он решает?
2. Что такое разреженность модели и какой метод регуляризации её обеспечивает?
3. Как влияет увеличение $\lambda$ на коэффициенты в гребневой регрессии?
4. Почему перед применением регуляризации необходимо стандартизировать признаки?
5. В чём преимущество Elastic Net перед чистым Lasso и чистым Ridge?

### Для профессионалов

1. Объясните байесовский смысл $L_2$- и $L_1$-регуляризации. Какие априорные распределения им соответствуют?
2. Почему $L_1$-штраф приводит к обнулению коэффициентов? Дайте геометрическую интерпретацию или поясните через условия субградиента.
3. Опишите идею координатного спуска для Lasso. Почему этот метод эффективен для $L_1$-регуляризованных задач?
4. Как связана гребневая регрессия с регуляризацией Тихонова?
5. Какие методы позволяют оценить неопределённость коэффициентов в Lasso, несмотря на смещённость оценок? Кратко поясните идею бутстрепа и дебайасинга.















